# OCKM-T: held-out large-scale benchmark (Google Colab, fast version)

This notebook runs OCKM-T, the circle (concentric-sector) tree, their combination OCKM-TC and the paper's baselines on **large data sets that were never used for tuning**: Covertype (581,012 points), CIFAR-10 and CIFAR-100 image features (50,000 points), and 20 Newsgroups and AG News sentence embeddings (18,846 and 120,000 points), with k from 4 to 1,000.

**Main question.** Does combining the line tree (OCKM-T) and the circle tree give better centres than spending the same work on OCKM-T alone? Compare *Line+Circle (2 runs)* with *Line+Line (2 runs)*, and *Circle tree* with *OCKM-T*, in cell 7 ('Circle tree' is OCKM-S in the paper), especially at large k.

**How to use**
1. Colab: `Runtime → Change runtime type → T4 GPU`. Kaggle: Settings → Accelerator → GPU T4, and Internet on.
2. Run the cells from top to bottom. On Colab, sign in to Google Drive; results are written to `MyDrive/OCKM-T_heldout`.
3. Every finished run is saved at once to Drive. If Colab disconnects, reconnect and run all cells again: finished runs are skipped.

**Why this version is faster**
* **Lloyd runs on the GPU** (`GPU_LLOYD = True`). It follows scikit-learn's Lloyd rules exactly; on test data it gave the same final cost as scikit-learn to within 1e-9. Lloyd was most of the running time.
* **OCKM-TC is not run separately.** It is built from the OCKM-T and circle-tree runs with the same seed.
* **Weaker baselines use fewer seeds** (`OTHER_RUNS = 3`). OCKM-T, the circle tree and greedy k-means++ keep all their seeds.
* **The work is split into 3 parts** (`PART` in cell 3) that can run at the same time in different sessions (Colab, Kaggle). Each part writes its own CSV file, and the results cells read them all.
* The same notebook runs on **Kaggle** without changes (cell 1 detects it). **No file needs to be uploaded or attached**: all the code is inside the notebook.

In [ ]:
# 1. Paths. Works on Google Colab and on Kaggle. No upload is needed: all code is inside this notebook.
import os
ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    WORK_DIR = '/kaggle/working/OCKM-T_heldout'          # download results from the Output tab when the run ends
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/OCKM-T_heldout'
os.makedirs(WORK_DIR, exist_ok=True)
# cached feature matrices (.npz). On Kaggle they are kept outside /kaggle/working so the saved Output stays small
CACHE_DIR = '/tmp/ockm_features' if ON_KAGGLE else os.path.join(WORK_DIR, 'features')
CODE_DIR = '/content/ockm_code'
os.makedirs(CODE_DIR, exist_ok=True)
print('running on', 'Kaggle' if ON_KAGGLE else 'Colab'); print('working folder:', WORK_DIR)

In [ ]:
# 2. Install packages
!pip -q install numba threadpoolctl sentence-transformers datasets

The next cells write the OCKM-T code files (unchanged from the ICML code, plus `ockm_tc.py`). Just run them.

In [ ]:
%%writefile /content/ockm_code/ockmt.py
"""OCKM-T: orbit-cut tree summaries for k-means seeding (compiled implementation).

Pipeline
1. Projection. Centre X and, when d > p, multiply by a fixed Gaussian matrix G (d x p).
2. Orbit-cut tree (deterministic). Starting from one cell, the cell with the largest SSE
   is split until m = gamma * k cells exist. For a cell with principal plane (u, v), the
   candidate cuts are hyperplanes whose normals form the orbit of the first principal
   direction under rotations of the principal plane, n_t = u cos t + v sin t,
   t = j pi / n_dirs, j = 0..n_dirs-1. Along each normal the offset is chosen by a
   binned scan of the SSE reduction in the leading r principal coordinates. The PCA-Part
   cut (t = 0, offset 0) is always a candidate. Among the candidates the cut with the
   largest EXACT SSE reduction n ||S_L||^2 / (n_L n_R) is chosen, and it is refined by a
   few 2-means steps inside the cell (each step moves to the bisector of the two means,
   so the SSE never increases).
3. Summary search. The m leaves form a weighted summary (centroid, count). Two starts on
   the summary: the k-leaf partition of the tree and one D^2 (k-means++) sample. Each
   start is improved by LocalSearch++ swaps and weighted Lloyd; the lower summary cost wins.
4. Lift and polish. Each summary cluster is mapped to the mean of the original points in
   its leaves, and a few weighted Lloyd steps on the full-dimensional leaf centroids remove
   the projection distortion.

Guarantee: E[cost(X, C)] <= (1 + 2 alpha) Phi + 2 alpha OPT_k(X), where Phi is the
within-leaf cost (returned in info) and alpha is the approximation factor of the
summary algorithm (see the paper, Section 4).
"""
from __future__ import annotations
import math, time
import numpy as np
import numba as nb
from scipy import sparse
from sklearn.cluster import kmeans_plusplus
from threadpoolctl import ThreadpoolController

_TPC = None


def _one_blas_thread():
    """Context that limits BLAS to one thread. The tree calls BLAS only on small matrices
    (p x p covariances, n_c x 4 products), where multi-threaded BLAS is slower and very slow
    when another process competes for the cores."""
    global _TPC
    if _TPC is None:
        _TPC = ThreadpoolController()
    return _TPC.limit(limits=1, user_api="blas")

P_DEFAULT = 64


def project(X, p=P_DEFAULT, seed=0):
    """Centred data in the working space: X - mean (d <= p) or (X - mean) G (d > p)."""
    X = np.asarray(X, dtype=np.float64)
    n, d = X.shape
    mu = X.mean(0)
    if d <= p:
        return np.ascontiguousarray(X - mu), None, mu
    G = np.random.default_rng(seed).standard_normal((d, p)) / math.sqrt(p)
    return np.ascontiguousarray(X @ G - mu @ G), G, mu


# ----------------------------------------------------------------------------- heap
@nb.njit(cache=True)
def _hpush(hk, hv, size, key, val):
    i = size
    hk[i] = key; hv[i] = val
    while i > 0:
        par = (i - 1) >> 1
        if hk[par] >= hk[i]:
            break
        hk[par], hk[i] = hk[i], hk[par]
        hv[par], hv[i] = hv[i], hv[par]
        i = par
    return size + 1


@nb.njit(cache=True)
def _hpop(hk, hv, size):
    key, val = hk[0], hv[0]
    size -= 1
    hk[0] = hk[size]; hv[0] = hv[size]
    i = 0
    while True:
        l = 2 * i + 1; r = l + 1; b = i
        if l < size and hk[l] > hk[b]:
            b = l
        if r < size and hk[r] > hk[b]:
            b = r
        if b == i:
            break
        hk[b], hk[i] = hk[i], hk[b]
        hv[b], hv[i] = hv[i], hv[b]
        i = b
    return key, val, size


# ----------------------------------------------------------------------------- tree
@nb.njit(cache=True)
def _lead_dirs(Cv, Q0, rr, iters):
    """Leading rr eigenvectors of the p x p PSD matrix Cv (columns, decreasing order).
    Exact (eigh) for p <= 16; otherwise block subspace iteration with block size
    b = Q0.shape[1] >= rr from the start Q0, followed by Rayleigh-Ritz."""
    p = Cv.shape[0]
    if p <= 16:
        ev, V = np.linalg.eigh(Cv)
        out = np.empty((p, rr))
        for q in range(rr):
            out[:, q] = V[:, p - 1 - q]
        return out, out
    Q = Q0.copy()
    for it in range(iters):
        Qn, _ = np.linalg.qr(np.dot(Cv, Q))
        Q = np.ascontiguousarray(Qn)
    H = np.dot(Q.T, np.dot(Cv, Q))
    ev, W = np.linalg.eigh(H)
    b = Q.shape[1]
    Wr = np.empty((b, b))
    for q in range(b):
        Wr[:, q] = W[:, b - 1 - q]
    full = np.dot(Q, Wr)
    out = np.ascontiguousarray(full[:, :rr])
    return out, full


@nb.njit(cache=True)
def _orbit_tree(Z, m, k_snap, n_dirs, nbins, r_scan, refine, cov_cap, pca_only, block=4, iters=6, fam=0):
    """Greedy divisive tree on the rows of Z (n x p). Returns leaf labels, leaf sums,
    counts and SSEs, the centroids of the k_snap-leaf partition, the number of leaves,
    and how often each candidate direction was chosen (last entry: PCA-Part cut)."""
    n, p = Z.shape
    Zs = Z.copy()
    perm = np.arange(n)
    start = np.zeros(m, np.int64); end = np.zeros(m, np.int64)
    csum = np.zeros((m, p)); cnt = np.zeros(m, np.int64); csse = np.zeros(m)
    bb = min(block, p)
    basis = np.zeros((m, p, bb))
    np.random.seed(12345)
    Q0, _ = np.linalg.qr(np.random.standard_normal((p, bb)))
    basis[0] = Q0
    used = np.zeros(n_dirs + 1, np.int64)
    end[0] = n; cnt[0] = n
    for i in range(n):
        for j in range(p):
            csum[0, j] += Zs[i, j]
    mu0 = csum[0] / n
    s0 = 0.0
    for i in range(n):
        for j in range(p):
            t = Zs[i, j] - mu0[j]
            s0 += t * t
    csse[0] = s0
    hk = np.zeros(2 * m + 2); hv = np.zeros(2 * m + 2, np.int64)
    hsize = _hpush(hk, hv, 0, s0, 0)
    tol = 1e-12 * max(s0, 1e-300)
    nleaves = 1
    snapC = np.zeros((k_snap, p))
    snap_done = False
    if k_snap == 1:
        snapC[0] = mu0
        snap_done = True
    r = min(r_scan, p)
    angs = np.zeros(n_dirs)
    for a in range(n_dirs):
        angs[a] = a * math.pi / n_dirs
    nfix = n_dirs + 4 if fam >= 1 else 0
    nscan = n_dirs if fam != 1 else 0
    ncand = 1 if pca_only else nscan + nfix + 1
    while nleaves < m and hsize > 0:
        key, j, hsize = _hpop(hk, hv, hsize)
        if key <= tol:
            break
        s = start[j]; e = end[j]; nc = e - s
        if nc < 2:
            continue
        mu = csum[j] / nc
        B = np.empty((nc, p))                              # centred rows of the cell
        sqB = np.empty(nc)
        for i in range(nc):
            acc = 0.0
            for jj in range(p):
                t = Zs[s + i, jj] - mu[jj]
                B[i, jj] = t
                acc += t * t
            sqB[i] = acc
        # principal directions (covariance of at most cov_cap rows, strided)
        if p == 1:
            Vr = np.ones((1, 1))
            rr = 1
        else:
            if nc > cov_cap:
                step = nc // cov_cap
                Bs = np.ascontiguousarray(B[::step])
            else:
                Bs = B
            if p <= 4:
                nb_ = Bs.shape[0]
                Cv = np.zeros((p, p))
                for i in range(nb_):
                    for j1 in range(p):
                        t = Bs[i, j1]
                        for j2 in range(j1, p):
                            Cv[j1, j2] += t * Bs[i, j2]
                for j1 in range(p):
                    for j2 in range(j1):
                        Cv[j1, j2] = Cv[j2, j1]
            else:
                Cv = np.dot(Bs.T, Bs)
            rr = r if not pca_only else 1
            rr = min(rr, bb)
            Vr, full = _lead_dirs(Cv, np.ascontiguousarray(basis[j]), rr, iters if nleaves > 1 else 5 * iters)
            if full.shape[1] == bb:
                basis[j] = full
        Y = np.dot(B, Vr)                                  # nc x rr leading principal coordinates
        # candidate cuts: (direction index, threshold); last one is the PCA-Part cut
        cdir = np.zeros(ncand, np.int64); cthr = np.zeros(ncand)
        cdir[ncand - 1] = 0; cthr[ncand - 1] = 0.0
        if (not pca_only) and nfix > 0:
            for a in range(n_dirs):
                cdir[nscan + a] = a; cthr[nscan + a] = 0.0
            u0 = np.sort(Y[:, 0])
            qs = (0.25, 0.375, 0.625, 0.75)
            for qi in range(4):
                cdir[nscan + n_dirs + qi] = 0
                cthr[nscan + n_dirs + qi] = u0[min(nc - 1, int(qs[qi] * (nc - 1)))]
        if (not pca_only) and nscan > 0:
            proj = np.empty(nc)
            bcnt = np.zeros(nbins); bsum = np.zeros((nbins, rr))
            for a in range(n_dirs):
                ca = math.cos(angs[a]); sa = math.sin(angs[a])
                lo = 1e300; hi = -1e300
                for i in range(nc):
                    val = Y[i, 0] * ca + (Y[i, 1] * sa if rr > 1 else 0.0)
                    proj[i] = val
                    if val < lo:
                        lo = val
                    if val > hi:
                        hi = val
                cdir[a] = a
                if hi - lo <= 0.0:
                    cthr[a] = 0.0
                    continue
                width = (hi - lo) / nbins
                bcnt[:] = 0.0; bsum[:, :] = 0.0
                for i in range(nc):
                    bi = int((proj[i] - lo) / width)
                    if bi >= nbins:
                        bi = nbins - 1
                    bcnt[bi] += 1.0
                    for q in range(rr):
                        bsum[bi, q] += Y[i, q]
                best = -1.0; bq = -1
                PL = np.zeros(rr); nL = 0.0
                for bi in range(nbins - 1):
                    nL += bcnt[bi]
                    for q in range(rr):
                        PL[q] += bsum[bi, q]
                    nR = nc - nL
                    if nL <= 0.0 or nR <= 0.0 or bcnt[bi] == 0.0:
                        continue
                    nn = 0.0
                    for q in range(rr):
                        nn += PL[q] * PL[q]
                    red = nc * nn / (nL * nR)
                    if red > best:
                        best = red; bq = bi
                cthr[a] = lo + (bq + 1) * width if bq >= 0 else 0.0
        # exact SSE reduction of every candidate
        nLc = np.zeros(ncand)
        ccos = np.empty(ncand); csin = np.empty(ncand)
        for c in range(ncand):
            ccos[c] = math.cos(angs[cdir[c]]); csin[c] = math.sin(angs[cdir[c]])
        if p <= 16:
            SL = np.zeros((ncand, p))
            M = np.zeros((1, ncand))
            for i in range(nc):
                y0 = Y[i, 0]; y1 = Y[i, 1] if rr > 1 else 0.0
                for c in range(ncand):
                    if y0 * ccos[c] + y1 * csin[c] > cthr[c]:
                        nLc[c] += 1.0
                        for jj in range(p):
                            SL[c, jj] += B[i, jj]
        else:
            M = np.zeros((nc, ncand))
            for i in range(nc):
                y0 = Y[i, 0]; y1 = Y[i, 1] if rr > 1 else 0.0
                for c in range(ncand):
                    if y0 * ccos[c] + y1 * csin[c] > cthr[c]:
                        M[i, c] = 1.0
                        nLc[c] += 1.0
            SL = np.dot(M.T, B)                            # ncand x p side sums
        bestc = -1; bestred = -1.0
        for c in range(ncand):
            nR = nc - nLc[c]
            if nLc[c] <= 0.0 or nR <= 0.0:
                continue
            nn = 0.0
            for jj in range(p):
                nn += SL[c, jj] * SL[c, jj]
            red = nc * nn / (nLc[c] * nR)
            if red > bestred + 1e-15 * abs(bestred):
                bestred = red; bestc = c
        if bestc < 0:
            continue                                       # cell cannot be split
        used[cdir[bestc] if bestc < ncand - 1 else n_dirs] += 1
        left = np.empty(nc, np.bool_)
        for i in range(nc):
            y0 = Y[i, 0]; y1 = Y[i, 1] if rr > 1 else 0.0
            left[i] = y0 * ccos[bestc] + y1 * csin[bestc] > cthr[bestc]
        sL = SL[bestc].copy(); nl = nLc[bestc]
        # 2-means refinement inside the cell
        for it in range(refine):
            nr = nc - nl
            dv = np.empty(p); thr = 0.0
            for jj in range(p):
                aL = sL[jj] / nl; aR = -sL[jj] / nr
                dv[jj] = aR - aL
                thr += 0.5 * (aR * aR - aL * aL)
            pr = np.dot(B, dv)
            nl2 = 0.0; changed = 0
            for i in range(nc):
                lf = pr[i] < thr
                if lf != left[i]:
                    changed += 1
                if lf:
                    nl2 += 1.0
            if changed == 0 or nl2 == 0.0 or nl2 == nc:
                break
            sL[:] = 0.0
            for i in range(nc):
                left[i] = pr[i] < thr
                if left[i]:
                    for jj in range(p):
                        sL[jj] += B[i, jj]
            nl = nl2
        # child statistics
        nlI = int(nl)
        sqL = 0.0; sqAll = 0.0
        for i in range(nc):
            sqAll += sqB[i]
            if left[i]:
                sqL += sqB[i]
        nn = 0.0
        for jj in range(p):
            nn += sL[jj] * sL[jj]
        sseL = max(sqL - nn / nl, 0.0)
        sseR = max((sqAll - sqL) - nn / (nc - nl), 0.0)
        # reorder the segment: left rows first
        tmpZ = Zs[s:e].copy(); tmpP = perm[s:e].copy()
        a0 = s; b0 = s + nlI
        for i in range(nc):
            if left[i]:
                Zs[a0] = tmpZ[i]; perm[a0] = tmpP[i]; a0 += 1
            else:
                Zs[b0] = tmpZ[i]; perm[b0] = tmpP[i]; b0 += 1
        nid = nleaves
        start[nid] = s + nlI; end[nid] = e; cnt[nid] = nc - nlI
        end[j] = s + nlI; cnt[j] = nlI
        for jj in range(p):
            full = csum[j, jj]
            csum[j, jj] = sL[jj] + nl * mu[jj]
            csum[nid, jj] = full - csum[j, jj]
        csse[j] = sseL; csse[nid] = sseR
        basis[nid] = basis[j]
        hsize = _hpush(hk, hv, hsize, sseL, j)
        hsize = _hpush(hk, hv, hsize, sseR, nid)
        nleaves += 1
        if (not snap_done) and nleaves == k_snap:
            for c in range(k_snap):
                snapC[c] = csum[c] / cnt[c]
            snap_done = True
    if not snap_done:
        for c in range(min(k_snap, nleaves)):
            snapC[c] = csum[c] / cnt[c]
    labels = np.empty(n, np.int64)
    for c in range(nleaves):
        for i in range(start[c], end[c]):
            labels[perm[i]] = c
    return labels, csum[:nleaves].copy(), cnt[:nleaves].copy(), csse[:nleaves].copy(), snapC, nleaves, used


# ----------------------------------------------------------------------------- distances
def _top2_blas(R, C, chunk=4096):
    """Nearest and second-nearest centre of every row of R (squared distances)."""
    m, k = len(R), len(C)
    a = np.zeros(m, np.int64); b = np.zeros(m, np.int64)
    d1 = np.zeros(m); d2 = np.full(m, np.inf)
    sqC = np.einsum("ij,ij->i", C, C)
    for i in range(0, m, chunk):
        Rb = R[i:i + chunk]
        D = np.maximum(np.einsum("ij,ij->i", Rb, Rb)[:, None] + sqC[None, :] - 2.0 * Rb @ C.T, 0.0)
        rr = np.arange(len(Rb))
        if k == 1:
            a[i:i + chunk] = 0; b[i:i + chunk] = 0; d1[i:i + chunk] = D[:, 0]
            continue
        part = np.argpartition(D, 1, axis=1)[:, :2]
        v0, v1 = D[rr, part[:, 0]], D[rr, part[:, 1]]
        sw = v1 < v0
        a[i:i + chunk] = np.where(sw, part[:, 1], part[:, 0]); b[i:i + chunk] = np.where(sw, part[:, 0], part[:, 1])
        d1[i:i + chunk] = np.minimum(v0, v1); d2[i:i + chunk] = np.maximum(v0, v1)
    return a, b, d1, d2


@nb.njit(cache=True)
def _ls_core(R, w, C, a, b, d1, d2, steps, seed):
    """LocalSearch++ (Lattanzi and Sohler, 2019) on weighted points. Swap costs for all k
    centres in O(m) per step from nearest / second-nearest bookkeeping. A point whose
    triangle-inequality lower bound |d(x, c_a) - d(r, c_a)| on its distance to the
    candidate x is at least sqrt(d2) is unaffected by the swap and is not evaluated."""
    np.random.seed(seed)
    m, p = R.shape; k = C.shape[0]
    cost = 0.0
    for i in range(m):
        cost += w[i] * d1[i]
    rc = np.zeros(k)                                   # removal cost of every centre
    for i in range(m):
        rc[a[i]] += w[i] * (d2[i] - d1[i])
    cum = np.empty(m); dx = np.empty(m); delta = np.empty(k); dcx = np.empty(k)
    sq1 = np.sqrt(d1)
    acc = 0
    for st in range(steps):
        tot = 0.0
        for i in range(m):
            tot += w[i] * d1[i]
            cum[i] = tot
        if tot <= 0.0:
            break
        x = np.searchsorted(cum, np.random.random() * tot, side="right")
        if x >= m:
            x = m - 1
        for q in range(k):
            t = 0.0
            for jj in range(p):
                df = R[x, jj] - C[q, jj]
                t += df * df
            dcx[q] = math.sqrt(t)
        keep = cost
        for q in range(k):
            delta[q] = rc[q]
        for i in range(m):
            lb = dcx[a[i]] - sq1[i]
            if lb < 0.0:
                lb = -lb
            if lb * lb >= d2[i]:
                dx[i] = np.inf                             # dx >= d2: contribution already in rc
                continue
            t = 0.0
            for jj in range(p):
                df = R[i, jj] - R[x, jj]
                t += df * df
            dx[i] = t
            if t < d1[i]:
                keep -= w[i] * (d1[i] - t)
                base = t
            else:
                base = d1[i]
            alt = t if t < d2[i] else d2[i]
            delta[a[i]] += w[i] * ((alt - base) - (d2[i] - d1[i]))
        c = 0
        for q in range(1, k):
            if delta[q] < delta[c]:
                c = q
        new = keep + delta[c]
        if new >= cost * (1.0 - 1e-12):
            continue
        acc += 1
        for jj in range(p):
            C[c, jj] = R[x, jj]
        for i in range(m):
            if a[i] == c or b[i] == c:
                f1 = np.inf; f2 = np.inf; i1 = 0; i2 = 0
                for q in range(k):
                    t = 0.0
                    for jj in range(p):
                        df = R[i, jj] - C[q, jj]
                        t += df * df
                    if t < f1:
                        f2 = f1; i2 = i1; f1 = t; i1 = q
                    elif t < f2:
                        f2 = t; i2 = q
                a[i] = i1; b[i] = i2; d1[i] = f1; d2[i] = f2
            elif dx[i] < d1[i]:
                b[i] = a[i]; d2[i] = d1[i]; a[i] = c; d1[i] = dx[i]
            elif dx[i] < d2[i]:
                b[i] = c; d2[i] = dx[i]
        cost = 0.0
        rc[:] = 0.0
        for i in range(m):
            cost += w[i] * d1[i]
            rc[a[i]] += w[i] * (d2[i] - d1[i])
            sq1[i] = math.sqrt(d1[i])
    return cost, acc


def local_search_pp(R, w, C, steps, seed=0):
    C = np.array(C, dtype=np.float64, copy=True)
    if steps <= 0 or len(C) < 2:
        a, b, d1, d2 = _top2_blas(R, C)
        return C, float(w @ d1), 0
    a, b, d1, d2 = _top2_blas(R, C)
    cost, acc = _ls_core(np.ascontiguousarray(R), w, C, a, b, d1, d2, int(steps), int(seed))
    return C, float(cost), int(acc)


@nb.njit(cache=True)
def _assign_small(R, C):
    m, p = R.shape; k = C.shape[0]
    lab = np.zeros(m, np.int64); dmin = np.zeros(m)
    for i in range(m):
        best = np.inf; bi = 0
        for q in range(k):
            t = 0.0
            for jj in range(p):
                df = R[i, jj] - C[q, jj]
                t += df * df
            if t < best:
                best = t; bi = q
        lab[i] = bi; dmin[i] = best
    return lab, dmin


def _assign(R, C, chunk=4096):
    m, p = R.shape; k = len(C)
    if m * k * p <= 2_000_000:
        return _assign_small(R, C)
    lab = np.empty(m, np.int64); dmin = np.empty(m)
    sqC = np.einsum("ij,ij->i", C, C)
    for i in range(0, m, chunk):
        Rb = R[i:i + chunk]
        D = sqC[None, :] - 2.0 * Rb @ C.T
        l = D.argmin(1)
        lab[i:i + chunk] = l
        dmin[i:i + chunk] = np.maximum(D[np.arange(len(Rb)), l] + np.einsum("ij,ij->i", Rb, Rb), 0.0)
    return lab, dmin


@nb.njit(cache=True)
def _wlloyd_small(R, w, C, max_iter):
    m, p = R.shape; k = C.shape[0]
    lab, dmin = _assign_small(R, C)
    S = np.zeros((k, p)); W = np.zeros(k)
    for it in range(max_iter):
        S[:, :] = 0.0; W[:] = 0.0
        for i in range(m):
            W[lab[i]] += w[i]
            for jj in range(p):
                S[lab[i], jj] += w[i] * R[i, jj]
        for q in range(k):
            if W[q] > 0:
                for jj in range(p):
                    C[q, jj] = S[q, jj] / W[q]
        lab2, dmin = _assign_small(R, C)
        same = True
        for i in range(m):
            if lab2[i] != lab[i]:
                same = False
                break
        lab = lab2
        if same:
            break
    cost = 0.0
    for i in range(m):
        cost += w[i] * dmin[i]
    return C, lab, cost


@nb.njit(cache=True)
def d2_seeding(R, w, k, seed):
    """Vanilla D^2 sampling (k-means++ without greedy trials) on weighted points. A point is
    skipped when the triangle inequality shows that the new centre cannot be closer."""
    np.random.seed(seed)
    m, p = R.shape
    C = np.empty((k, p))
    cum = np.cumsum(w)
    x = np.searchsorted(cum, np.random.random() * cum[-1], side="right")
    x = min(x, m - 1)
    C[0] = R[x]
    d = np.empty(m); a = np.zeros(m, np.int64); dcc = np.empty(k)
    for i in range(m):
        t = 0.0
        for jj in range(p):
            df = R[i, jj] - C[0, jj]
            t += df * df
        d[i] = t
    for c in range(1, k):
        tot = 0.0
        for i in range(m):
            tot += w[i] * d[i]
            cum[i] = tot
        if tot <= 0.0:
            x = c % m
        else:
            x = np.searchsorted(cum, np.random.random() * tot, side="right")
            x = min(x, m - 1)
        C[c] = R[x]
        for q in range(c):
            t = 0.0
            for jj in range(p):
                df = C[c, jj] - C[q, jj]
                t += df * df
            dcc[q] = t
        for i in range(m):
            if dcc[a[i]] >= 4.0 * d[i]:
                continue
            t = 0.0
            for jj in range(p):
                df = R[i, jj] - C[c, jj]
                t += df * df
            if t < d[i]:
                d[i] = t; a[i] = c
    return C


@nb.njit(cache=True)
def _top2_small(R, C):
    m, p = R.shape; k = C.shape[0]
    a = np.zeros(m, np.int64); b = np.zeros(m, np.int64); d1 = np.empty(m); d2 = np.empty(m)
    for i in range(m):
        f1 = np.inf; f2 = np.inf; i1 = 0; i2 = 0
        for q in range(k):
            t = 0.0
            for jj in range(p):
                df = R[i, jj] - C[q, jj]
                t += df * df
            if t < f1:
                f2 = f1; i2 = i1; f1 = t; i1 = q
            elif t < f2:
                f2 = t; i2 = q
        a[i] = i1; b[i] = i2; d1[i] = f1; d2[i] = f2
    return a, b, d1, d2


@nb.njit(cache=True)
def _summary_search_small(R, w, Ctree, S, steps, max_iter, seed, ls_d2):
    """All summary starts in compiled code (small problems): start 0 is the tree start,
    starts 1..S-1 are D^2 samples; each gets LocalSearch++ (the D^2 starts only if ls_d2)
    and weighted Lloyd. Returns the best centres, their leaf labels, and all costs."""
    k = Ctree.shape[0]
    costs = np.empty(S)
    bestC = Ctree.copy(); bestlab = np.zeros(R.shape[0], np.int64); bestcost = np.inf
    for si in range(S):
        if si == 0:
            C0 = Ctree.copy()
        else:
            C0 = d2_seeding(R, w, k, seed + 1 + 7919 * si)
        if steps > 0 and k > 1 and (si == 0 or ls_d2):
            a, b, d1, d2 = _top2_small(R, C0)
            _ls_core(R, w, C0, a, b, d1, d2, steps, seed + 2 + 7919 * si)
        C1, lab, cst = _wlloyd_small(R, w, C0, max_iter)
        costs[si] = cst
        if cst < bestcost:
            bestcost = cst; bestC = C1.copy(); bestlab = lab.copy()
    return bestC, bestlab, bestcost, costs


def weighted_lloyd(R, w, C, max_iter=100):
    """Weighted Lloyd on (R, w) from C until the assignment is stable or max_iter updates.
    Empty clusters keep their centre. Returns centres, labels and cost of the returned
    centres (the labels are the assignment to the returned centres)."""
    R = np.ascontiguousarray(R); C = np.array(C, dtype=np.float64, copy=True)
    k = len(C)
    if R.shape[0] * k * R.shape[1] <= 2_000_000:
        return _wlloyd_small(R, np.ascontiguousarray(w, dtype=np.float64), C, int(max_iter))
    lab, dmin = _assign(R, C)
    for it in range(max_iter):
        W = np.bincount(lab, weights=w, minlength=k)
        S = sparse.csr_matrix((w, (lab, np.arange(len(R)))), shape=(k, len(R))) @ R
        nz = W > 0
        C[nz] = S[nz] / W[nz, None]
        lab2, dmin = _assign(R, C)
        if np.array_equal(lab2, lab):
            lab = lab2
            break
        lab = lab2
    return C, lab, float(w @ dmin)


# ----------------------------------------------------------------------------- main
def _n_starts(n, k, m, n_starts, max_starts):
    """Number of summary starts. 'auto': the summary search gets about the work of the
    tree, S * k * m <= 64 n, with 2 <= S <= max_starts."""
    if n_starts == "auto":
        return int(min(max_starts, max(2, (64 * n) // max(1, k * m))))
    return int(n_starts)


def summary_search(R, w, Ctree, S, steps, max_iter=100, seed=0, use_tree=True):
    """Weighted k-means on the summary (R, w). Start 0 is the tree start (if use_tree),
    the other starts are D^2 samples; every start gets `steps` LocalSearch++ swaps and
    weighted Lloyd. Returns the centres, leaf labels and cost of the best start, and the
    costs of all starts."""
    R = np.ascontiguousarray(R, dtype=np.float64); w = np.ascontiguousarray(w, dtype=np.float64)
    k = len(Ctree)
    if use_tree and R.shape[0] * k * R.shape[1] <= 2_000_000:
        return _summary_search_small(R, w, np.ascontiguousarray(Ctree), S, steps, int(max_iter), int(seed), True)
    costs = np.empty(S); best = None
    for si in range(S):
        C0 = np.array(Ctree, dtype=np.float64) if (si == 0 and use_tree) else d2_seeding(R, w, k, seed + 1 + 7919 * si)
        if steps > 0 and k > 1:
            C0, _, _ = local_search_pp(R, w, C0, steps, seed=seed + 2 + 7919 * si)
        Cs, lab, cst = weighted_lloyd(R, w, C0, max_iter=max_iter)
        costs[si] = cst
        if best is None or cst < best[2]:
            best = (Cs, lab, cst)
    return best[0], best[1], best[2], costs


def ockm_t(X, k, **kw):
    """OCKM-T seeding; see _ockm_t_impl for the parameters. Small problems run with one BLAS
    thread throughout (their matrix products are too small to gain from threads)."""
    X = np.asarray(X, dtype=np.float64)
    if X.shape[0] * X.shape[1] * min(X.shape[1], kw.get("p", P_DEFAULT)) < 5e7:
        with _one_blas_thread():
            return _ockm_t_impl(X, k, **kw)
    return _ockm_t_impl(X, k, **kw)


def _ockm_t_impl(X, k, gamma=8, p=P_DEFAULT, n_dirs=4, nbins=64, r_scan=2, refine=2, cov_cap=4096, block=4, iters=6,
           pca_only=False, ls_steps=2.0, n_starts="auto", max_starts=16, use_tree=True, max_iter=100,
           polish_budget=5e9, seed=0, return_info=False):
    """OCKM-T seeding. Returns a k x d array of centres (and an info dict).

    gamma: leaves per centre (m = gamma * k); p: projection dimension (used when d > p);
    n_dirs: rotations of the principal direction searched per split; nbins: bins of the
    offset scan; refine: 2-means steps per split; pca_only: PCA-Part cuts only (ablation);
    ls_steps: LocalSearch++ swaps per start, as a multiple of k; n_starts: summary starts
    ('auto' or an integer); use_tree: include the tree start; polish_budget: flops allowed
    for the weighted Lloyd polish on the full-dimensional leaf centroids (0 disables)."""
    tt = {}
    t0 = time.perf_counter()
    X = np.asarray(X, dtype=np.float64)
    n, d = X.shape
    k = int(min(k, n))
    Z, G, mu = project(X, p, seed)
    tt["project"] = time.perf_counter() - t0; t0 = time.perf_counter()
    m = int(min(n, max(k, round(gamma * k))))
    with _one_blas_thread():
        labels, csum, cnt, csse, snapC, L, used = _orbit_tree(Z, m, k, n_dirs, nbins, r_scan, 0 if pca_only else refine,
                                                              cov_cap, pca_only, block, iters, 0)
    tt["tree"] = time.perf_counter() - t0; t0 = time.perf_counter()
    w = cnt.astype(np.float64)
    R = csum / w[:, None]
    phi = float(csse.sum())
    kk = min(k, L)
    S = _n_starts(n, kk, L, n_starts, max_starts)
    steps = int(round(ls_steps * kk)) if kk > 1 else 0
    Cs, lab_leaf, scost, costs = summary_search(R, w, snapC[:kk], S, steps, max_iter, seed, use_tree)
    tt["summary"] = time.perf_counter() - t0; t0 = time.perf_counter()
    # lift: mean of the original points in the leaves of each summary cluster
    if G is None:
        Sfull = csum + w[:, None] * mu[None, :]           # leaf sums in the original space
    else:
        Sfull = np.asarray(sparse.csr_matrix((np.ones(n), (labels, np.arange(n))), shape=(L, n)) @ X)
    num = np.asarray(sparse.csr_matrix((np.ones(L), (lab_leaf, np.arange(L))), shape=(kk, L)) @ Sfull)
    den = np.bincount(lab_leaf, weights=w, minlength=kk)
    C = np.empty((k, d))
    okc = den > 0
    C[:kk][okc] = num[okc] / den[okc, None]
    if (~okc).any() or kk < k:
        order = np.argsort(-w)
        fill = [i for i in range(k) if i >= kk or not okc[i]]
        C[fill] = (Sfull / w[:, None])[order[:len(fill)]]
    polish_it = 0
    if polish_budget > 0 and G is not None:
        polish_it = int(max(1, min(50, polish_budget // max(1, L * k * d))))
        C, _, _ = weighted_lloyd(Sfull / w[:, None], w, C, max_iter=polish_it)
    tt["lift"] = time.perf_counter() - t0
    if return_info:
        if G is None:
            phi_full = phi
        else:                                            # within-leaf cost in the original space (one pass)
            tot = float(np.einsum("ij,ij->", X, X) - n * mu @ mu)
            D = Sfull - w[:, None] * mu[None, :]
            phi_full = max(tot - float(np.einsum("ij,ij->i", D, D) @ (1.0 / w)), 0.0)
        wi = int(np.argmin(costs))
        names = [("tree" if (i == 0 and use_tree) else f"D2_{i}") for i in range(S)]
        return C, dict(m=L, phi=phi, phi_full=phi_full, summary_cost=float(scost), winner=names[wi], n_starts=S, polish_it=polish_it,
                       cand_costs={nm: float(c) for nm, c in zip(names, costs)},
                       cuts={f"dir{j}": int(used[j]) for j in range(n_dirs)} | {"pca": int(used[n_dirs])},
                       times=tt)
    return C


def warmup():
    """Compile the numba kernels once (they are cached on disk afterwards)."""
    rng = np.random.default_rng(0)
    X = rng.standard_normal((300, 5))
    ockm_t(X, 4)
    ockm_t(X, 4, pca_only=True)
    ockm_t(X, 4, n_starts=1, ls_steps=0, max_iter=0)
    ockm_t(rng.standard_normal((300, 80)), 4)
    ockm_t(rng.standard_normal((3000, 20)), 60)          # large-summary path


In [ ]:
%%writefile /content/ockm_code/ockm_tc.py
"""OCKM-TC: OCKM-T with a second, concentric-sector (circle) tree.

Two partitions of the data are built:
  * the orbit-cut tree of OCKM-T (straight cuts: rotated principal cuts + PCA-Part, 2-means refinement);
  * the concentric-sector tree of CCKM-S (each split is either a shell cut, i.e. a sphere about
    the cell centroid, or a sector cut, i.e. a threshold on the first principal coordinate,
    whichever gives the larger SSE reduction; no refinement, so shell cuts stay spheres).
The same summary search (tree start + D^2 starts, LocalSearch++, weighted Lloyd) and the same lift
are run on each summary, and the centres with the lower cost on X are returned.

Guarantee: each branch satisfies E[cost] <= (1+2a) Phi_b + 2a OPT_k (in the working space);
for d <= p both trees are deterministic, so the returned centres satisfy
E[cost] <= (1+2a) min(Phi_1, Phi_2) + 2a OPT_k.
"""
from __future__ import annotations
import time
import numpy as np
import numba as nb
from scipy import sparse
import ockmt
from ockmt import ockm_t, project, summary_search, weighted_lloyd, _n_starts, _one_blas_thread, P_DEFAULT


# ------------------------------------------------------------------ concentric-sector tree
@nb.njit(cache=True)
def _best_split(keys, B, sq, ncand):
    """Best threshold split of the rows of B (centred) in the order of keys.
    Gain of a split = n ||S_L||^2 / (n_L n_R), S_L the sum of the left centred rows.
    Returns (gain, number of rows on the left, order)."""
    nc, p = B.shape
    order = np.argsort(keys, kind="mergesort")
    run = np.zeros(p)
    best_g = -1.0; best_c = 0
    nxt = 0
    # candidate positions: ncand evenly spaced split positions in 1..nc-1
    cands = np.empty(ncand, np.int64)
    for t in range(ncand):
        cands[t] = 1 + (t * (nc - 2)) // max(1, ncand - 1) if ncand > 1 else nc // 2
    c_idx = 0
    for i in range(nc - 1):
        r = order[i]
        for j in range(p):
            run[j] += B[r, j]
        pos = i + 1
        while c_idx < ncand and cands[c_idx] < pos:
            c_idx += 1
        if c_idx < ncand and cands[c_idx] == pos:
            # skip ties in the key: the cut must separate different key values
            if keys[order[i]] == keys[order[i + 1]]:
                continue
            s2 = 0.0
            for j in range(p):
                s2 += run[j] * run[j]
            g = nc * s2 / (pos * (nc - pos))
            if g > best_g:
                best_g = g; best_c = pos
    return best_g, best_c, order


@nb.njit(cache=True)
def _lead_dir(B):
    """First principal direction of the centred rows B (power iteration, 60 steps)."""
    nc, p = B.shape
    if p == 1:
        return np.ones(1)
    C = B.T @ B
    v = np.ones(p) / np.sqrt(p)
    for j in range(p):                      # deterministic start: largest-variance axis
        if C[j, j] > C[0, 0]:
            pass
    jmax = 0
    for j in range(p):
        if C[j, j] > C[jmax, jmax]:
            jmax = j
    v[:] = 0.0; v[jmax] = 1.0
    for _ in range(60):
        w = C @ v
        nrm = np.sqrt((w * w).sum())
        if nrm == 0.0:
            break
        v = w / nrm
    return v


@nb.njit(cache=True)
def _cs_tree(Z, m, k_snap, ncand, use_shell, use_sector):
    n, p = Z.shape
    Zs = Z.copy()
    start = np.zeros(m, np.int64); end = np.zeros(m, np.int64)
    csum = np.zeros((m, p)); cnt = np.zeros(m, np.int64); csse = np.zeros(m)
    labels = np.zeros(n, np.int64)
    perm = np.arange(n)
    end[0] = n; cnt[0] = n
    for i in range(n):
        for j in range(p):
            csum[0, j] += Zs[i, j]
    mu0 = csum[0] / n
    s0 = 0.0
    for i in range(n):
        for j in range(p):
            t = Zs[i, j] - mu0[j]; s0 += t * t
    csse[0] = s0
    L = 1
    snapC = np.zeros((k_snap, p))
    if k_snap == 1:
        snapC[0] = mu0
    used = np.zeros(2, np.int64)                      # [shell, sector]
    frozen = np.zeros(m, np.bool_)
    tol = 1e-12 * max(s0, 1e-300)
    while L < m:
        # cell with the largest SSE (linear scan)
        jb = -1; kb = -1.0
        for c in range(L):
            if (not frozen[c]) and csse[c] > kb:
                kb = csse[c]; jb = c
        if jb < 0 or kb <= tol:
            break
        s = start[jb]; e = end[jb]; nc = e - s
        if nc < 2:
            frozen[jb] = True; continue
        mu = csum[jb] / nc
        B = np.empty((nc, p)); sq = np.empty(nc)
        for i in range(nc):
            a = 0.0
            for j in range(p):
                t = Zs[s + i, j] - mu[j]; B[i, j] = t; a += t * t
            sq[i] = a
        bg = -1.0; bc = 0; border = np.arange(nc); kind = -1
        if use_shell:
            g, c, o = _best_split(sq, B, sq, ncand)
            if g > bg:
                bg = g; bc = c; border = o; kind = 0
        if use_sector:
            u = _lead_dir(B)
            g, c, o = _best_split(B @ u, B, sq, ncand)
            if g > bg:
                bg = g; bc = c; border = o; kind = 1
        if bg <= 0.0 or bc <= 0 or bc >= nc:
            frozen[jb] = True; continue
        used[kind] += 1
        # reorder the cell: first bc rows (in key order) form the left child
        tmpZ = Zs[s:e].copy(); tmpP = perm[s:e].copy()
        for i in range(nc):
            Zs[s + i] = tmpZ[border[i]]; perm[s + i] = tmpP[border[i]]
        new = L
        start[new] = s + bc; end[new] = e; end[jb] = s + bc
        for c2 in (jb, new):
            a = start[c2]; b = end[c2]
            sm = np.zeros(p)
            for i in range(a, b):
                for j in range(p):
                    sm[j] += Zs[i, j]
            ncc = b - a
            mc = sm / ncc
            ss = 0.0
            for i in range(a, b):
                for j in range(p):
                    t = Zs[i, j] - mc[j]; ss += t * t
            csum[c2] = sm; cnt[c2] = ncc; csse[c2] = ss
        L += 1
        if L == k_snap:
            for c2 in range(L):
                snapC[c2] = csum[c2] / cnt[c2]
    for c in range(L):
        for i in range(start[c], end[c]):
            labels[perm[i]] = c
    return labels, csum[:L].copy(), cnt[:L].copy(), csse[:L].copy(), snapC, L, used


def _branch(X, k, labels, csum, cnt, csse, snapC, L, G, mu, seed, ls_steps, n_starts, max_starts, max_iter, polish_budget):
    """Summary search + lift for one tree (same code path as OCKM-T)."""
    n, d = X.shape
    w = cnt.astype(np.float64)
    R = csum / w[:, None]
    kk = min(k, L)
    S = _n_starts(n, kk, L, n_starts, max_starts)
    steps = int(round(ls_steps * kk)) if kk > 1 else 0
    Cs, lab_leaf, scost, costs = summary_search(R, w, snapC[:kk], S, steps, max_iter, seed, True)
    if G is None:
        Sfull = csum + w[:, None] * mu[None, :]
    else:
        Sfull = np.asarray(sparse.csr_matrix((np.ones(n), (labels, np.arange(n))), shape=(L, n)) @ X)
    num = np.asarray(sparse.csr_matrix((np.ones(L), (lab_leaf, np.arange(L))), shape=(kk, L)) @ Sfull)
    den = np.bincount(lab_leaf, weights=w, minlength=kk)
    C = np.empty((k, d)); okc = den > 0
    C[:kk][okc] = num[okc] / den[okc, None]
    if (~okc).any() or kk < k:
        order = np.argsort(-w); fill = [i for i in range(k) if i >= kk or not okc[i]]
        C[fill] = (Sfull / w[:, None])[order[:len(fill)]]
    if polish_budget > 0 and G is not None:
        it = int(max(1, min(50, polish_budget // max(1, L * k * d))))
        C, _, _ = weighted_lloyd(Sfull / w[:, None], w, C, max_iter=it)
    return C, float(csse.sum()), float(scost)


def data_cost(X, C, chunk=8192):
    cc = (C * C).sum(1); tot = 0.0
    for i in range(0, len(X), chunk):
        A = X[i:i + chunk]
        D = (A * A).sum(1)[:, None] + cc[None, :] - 2.0 * A @ C.T
        tot += float(np.maximum(D.min(1), 0).sum())
    return tot


def circle_tree_seeding(X, k, gamma=8, p=P_DEFAULT, ncand=32, use_shell=True, use_sector=True, ls_steps=2.0,
                        n_starts="auto", max_starts=16, max_iter=100, polish_budget=5e9, seed=0, return_info=False):
    """Seeding with the concentric-sector (circle) tree and the OCKM-T summary search."""
    X = np.asarray(X, dtype=np.float64); n, d = X.shape; k = int(min(k, n))
    t0 = time.perf_counter()
    Z, G, mu = project(X, p, seed)
    m = int(min(n, max(k, round(gamma * k))))
    with _one_blas_thread():
        labels, csum, cnt, csse, snapC, L, used = _cs_tree(Z, m, k, ncand, use_shell, use_sector)
    t_tree = time.perf_counter() - t0
    C, phi, scost = _branch(X, k, labels, csum, cnt, csse, snapC, L, G, mu, seed, ls_steps, n_starts, max_starts, max_iter, polish_budget)
    if return_info:
        return C, dict(phi=phi, summary_cost=scost, m=L, shell_cuts=int(used[0]), sector_cuts=int(used[1]), t_tree=t_tree)
    return C


def ockm_tc(X, k, seed=0, return_info=False, **kw):
    """OCKM-TC: run OCKM-T and the circle-tree seeding, return the centres with the lower cost on X."""
    X = np.asarray(X, dtype=np.float64)
    t0 = time.perf_counter()
    C1, i1 = ockm_t(X, k, seed=seed, return_info=True, **kw)
    t1 = time.perf_counter() - t0; t0 = time.perf_counter()
    C2, i2 = circle_tree_seeding(X, k, seed=seed, return_info=True)
    t2 = time.perf_counter() - t0
    c1, c2 = data_cost(X, C1), data_cost(X, C2)
    win = "circle" if c2 < c1 else "line"
    C = C2 if c2 < c1 else C1
    if return_info:
        return C, dict(winner=win, cost_line=c1, cost_circle=c2, phi_line=i1["phi"], phi_circle=i2["phi"],
                       t_line=t1, t_circle=t2, shell_cuts=i2["shell_cuts"], sector_cuts=i2["sector_cuts"])
    return C


def warmup():
    ockmt.warmup()
    rng = np.random.default_rng(0)
    for X, k in ((rng.standard_normal((300, 2)), 4), (rng.standard_normal((300, 5)), 4), (rng.standard_normal((300, 80)), 4),
                 (rng.standard_normal((3000, 20)), 60)):
        ockm_tc(X, k)


In [ ]:
%%writefile /content/ockm_code/qkmeans.py
"""Qkmeans seeding (Shah, Agrawal & Jaiswal, "Fast k-means Seeding Under The
Manifold Hypothesis", arXiv:2602.01104), reimplemented from Algorithms 1-3 of
the paper. No official code was found, so this is our own implementation.

Algorithm 1 PREPROCESS
    JL-type random projection to D' = ceil(ln(k/eps)/eps^2) dimensions
    (eps = eps_jl in (0, 1/4); the paper gives D' only up to a constant,
    we use constant 1 and skip the projection when D' >= D), centre the data,
    store ||x||^2 and ||X||_F^2.
Algorithm 2 SAMPLE(L, C, m)
    s ~ Uniform(X)                                   (fallback)
    repeat at most ceil(m ln k) times:
        x ~ kappa(.|C) = (||x||^2 + ||c1||^2) / (||X||_F^2 + n ||c1||^2)
        R ~ U(0,1)
        r = cost(x, QUERY(L, x)) / (2 rho^-1 (||x||^2 + ||c1||^2))
        if R <= r: return x
    return s
Algorithm 3 QKMEANS
    c1 ~ Uniform(X); then k-1 calls of SAMPLE.

QUERY(L, x) is the approximate nearest-centre search. The paper uses an LSH
structure that returns a centre within 1/sqrt(rho) of the true nearest
distance. Here QUERY is exact (rho = 1). With rho = 1 every accepted point is
drawn exactly from the D^2 (k-means++) distribution in the projected space, so
this variant gives Qkmeans its best possible seeding quality; its runtime is
that of our NumPy code, not of the authors' LSH implementation.

Proposals are drawn in blocks. The centre set does not change until a proposal
is accepted, so taking the first accepted proposal of a block gives exactly the
same distribution as drawing proposals one at a time.
"""
from __future__ import annotations
import math
import numpy as np


def qkmeans(X: np.ndarray, k: int, m: float = 10.0, rho: float = 1.0,
            eps_jl: float = 0.2, project: bool = True, seed=None,
            return_info: bool = False):
    rng = np.random.default_rng(seed)
    n, D = X.shape
    # ---------------- Algorithm 1: PREPROCESS ----------------
    Dp = int(math.ceil(math.log(k / eps_jl) / eps_jl ** 2))
    if project and Dp < D:
        G = rng.standard_normal((D, Dp)) / math.sqrt(Dp)
        Y = X @ G
    else:
        Dp = D
        Y = np.array(X, dtype=float, copy=True)
    Y -= Y.mean(axis=0)
    sq = np.einsum("ij,ij->i", Y, Y)
    F = float(sq.sum())
    cdf_sq = np.cumsum(sq); cdf_sq /= cdf_sq[-1]
    # ---------------- Algorithm 3: QKMEANS ----------------
    idx = [int(rng.integers(n))]
    c1sq = float(sq[idx[0]])
    p_norm = F / (F + n * c1sq)          # kappa is a mixture: ||x||^2-weighted w.p. p_norm, uniform otherwise
    Cmat = np.empty((k, Dp)); Cmat[0] = Y[idx[0]]
    Csq = np.empty(k); Csq[0] = c1sq
    T = (math.inf if math.isinf(m) else int(math.ceil(m * math.log(k)))) if k > 1 else 0
    proposals = 0; fallbacks = 0
    for t in range(1, k):
        s = int(rng.integers(n))           # Algorithm 2, step 1
        done = 0; chosen = None; B = 16
        while done < T:
            b = int(min(B, T - done, 1 << 16))
            u = rng.random(b)
            from_norm = rng.random(b) < p_norm
            x = np.where(from_norm, np.searchsorted(cdf_sq, u, side="right"), rng.integers(n, size=b))
            x = np.minimum(x, n - 1)
            R = rng.random(b)
            Yx = Y[x]
            d2 = sq[x][:, None] + Csq[None, :t] - 2.0 * Yx @ Cmat[:t].T
            cost = np.maximum(d2.min(axis=1), 0.0)
            r = cost / (2.0 / rho * (sq[x] + c1sq))
            acc = np.flatnonzero(R <= r)
            if acc.size:
                j = int(acc[0]); chosen = int(x[j]); proposals += j + 1; break
            proposals += b; done += b; B = min(B * 2, 1 << 16)
        if chosen is None:
            chosen = s; fallbacks += 1
        idx.append(chosen)
        Cmat[t] = Y[chosen]; Csq[t] = sq[chosen]
    idx = np.asarray(idx)
    if return_info:
        return X[idx].copy(), dict(idx=idx, Dp=Dp, proposals=proposals, fallbacks=fallbacks, T=T)
    return X[idx].copy()


In [ ]:
%%writefile /content/ockm_code/baselines2.py
"""Randomised seeding baselines reimplemented from their papers.

afkmc2        Bachem, Lucic, Hassani and Krause (NeurIPS 2016), "Fast and provably
              good seedings for k-means": proposal q(x) = 1/2 d(x,c1)^2 / sum + 1/(2n),
              Metropolis-Hastings chain of length m per centre (m = 200 as in the paper).
kmeans_par    Bahmani et al. (VLDB 2012), "Scalable k-means++": r = 5 rounds of
              oversampling with l = 2k, candidates weighted by cluster size, reduced
              to k by weighted k-means++ and a few weighted Lloyd steps.
"""
import numpy as np


def _mind2(A, C):
    d = np.einsum("ij,ij->i", A, A)[:, None] + np.einsum("ij,ij->i", C, C)[None, :] - 2.0 * A @ C.T
    return np.maximum(d.min(1), 0.0)


def _mind2_chunked(A, C, chunk=4096):
    out = np.empty(len(A))
    if len(C) == 0:
        out[:] = np.inf
        return out
    sqC = np.einsum("ij,ij->i", C, C)
    for i in range(0, len(A), chunk):
        B = A[i:i + chunk]
        out[i:i + chunk] = np.maximum((np.einsum("ij,ij->i", B, B)[:, None] + sqC[None, :] - 2.0 * B @ C.T).min(1), 0.0)
    return out


def _argmin_chunked(A, C, chunk=4096):
    out = np.empty(len(A), dtype=np.int64)
    sqC = np.einsum("ij,ij->i", C, C)
    for i in range(0, len(A), chunk):
        B = A[i:i + chunk]
        out[i:i + chunk] = (np.einsum("ij,ij->i", B, B)[:, None] + sqC[None, :] - 2.0 * B @ C.T).argmin(1)
    return out


def afkmc2(X, k, chain=200, seed=0):
    rng = np.random.default_rng(seed)
    n = len(X)
    c1 = int(rng.integers(n))
    d1 = np.maximum(np.einsum("ij,ij->i", X, X) - 2.0 * X @ X[c1] + X[c1] @ X[c1], 0.0)
    q = 0.5 * d1 / d1.sum() + 0.5 / n
    C = [X[c1]]
    for _ in range(1, k):
        cand = rng.choice(n, size=chain, p=q)
        dy = _mind2(X[cand], np.array(C))
        u = rng.random(chain)
        x, dx, qx = cand[0], dy[0], q[cand[0]]
        for j in range(1, chain):
            qy = q[cand[j]]
            if dx * qy == 0 or u[j] < (dy[j] * qx) / (dx * qy):
                x, dx, qx = cand[j], dy[j], qy
        C.append(X[x])
    return np.array(C)


def kmeans_par(X, k, seed=0, l=None, rounds=5):
    rng = np.random.default_rng(seed)
    n = len(X); l = 2 * k if l is None else l
    C = X[[int(rng.integers(n))]]
    d = _mind2_chunked(X, C)
    for r in range(rounds):
        if r > 0:
            d = np.minimum(d, _mind2_chunked(X, new))       # incremental update (same values)
        p = np.minimum(1.0, l * d / d.sum())
        new = X[rng.random(n) < p]
        C = np.vstack([C, new])
    C = np.unique(C, axis=0)
    while len(C) < k:
        d = _mind2_chunked(X, C); C = np.vstack([C, X[[int(np.argmax(d))]]])
    w = np.bincount(_argmin_chunked(X, C), minlength=len(C)).astype(float)
    keep = w > 0
    C, w = C[keep], w[keep]
    if len(C) <= k:
        return np.vstack([C, X[rng.choice(n, k - len(C), replace=False)]]) if len(C) < k else C
    idx = [int(rng.choice(len(C), p=w / w.sum()))]
    dmin = _mind2(C, C[idx])
    while len(idx) < k:
        p = w * dmin; j = int(rng.choice(len(C), p=p / p.sum())); idx.append(j)
        dmin = np.minimum(dmin, _mind2(C, C[[j]]))
    cen = C[idx].copy()
    for _ in range(10):
        a = (np.einsum("ij,ij->i", C, C)[:, None] + np.einsum("ij,ij->i", cen, cen)[None, :] - 2.0 * C @ cen.T).argmin(1)
        for j in range(k):
            msk = a == j
            if msk.any():
                cen[j] = (C[msk] * w[msk, None]).sum(0) / w[msk].sum()
    return cen


In [ ]:
%%writefile /content/ockm_code/det_baselines.py
"""Deterministic divisive seedings of Su and Dy (2007), "In search of deterministic
methods for initializing K-means and Gaussian mixture clustering", Intelligent
Data Analysis 11(4):319-338, as usually described (e.g. Celebi, Kingravi and
Vela 2013): start with one cluster; repeatedly split the cluster with the
largest SSE by a hyperplane through its centroid,
  PCA-Part: orthogonal to the cluster's principal direction,
  Var-Part: orthogonal to the coordinate axis of largest variance;
stop at k clusters and return their centroids.

Implementation: each split gathers the cluster once (O(n_c d)); SSEs come from
precomputed squared norms. The principal direction is found by randomised
subspace iteration (3 steps, fixed seed) on at most 10,000 points of the
cluster, the same routine OCKM-fast uses, so the result is deterministic.
"""
import numpy as np
from ockm_fast import rand_pcs


def divisive_part(X, k, mode="pca", n_pca=10000, seed=0):
    n = len(X)
    sq = np.einsum("ij,ij->i", X, X)
    rng = np.random.default_rng(seed)
    mu0 = X.mean(0)
    clusters, mus, sse = [np.arange(n)], [mu0], [float(sq.sum() - n * mu0 @ mu0)]
    while len(clusters) < k:
        j = int(np.argmax(sse))
        if sse[j] <= 0:
            break
        idx = clusters[j]; Xi = X[idx]; mu = mus[j]
        if mode == "pca":
            if len(idx) > n_pca:
                Xs = Xi[rng.choice(len(idx), n_pca, replace=False)]
            else:
                Xs = Xi
            v = rand_pcs(Xs, mu, 1, seed=seed, iters=3)[0]
            proj = Xi @ v - mu @ v
        else:
            var = np.einsum("ij,ij->j", Xi, Xi) / len(idx) - mu * mu
            dim = int(np.argmax(var))
            proj = Xi[:, dim] - mu[dim]
        left = proj <= 0
        nl = int(left.sum())
        if nl == 0 or nl == len(idx):
            sse[j] = -1.0            # cannot be split; never chosen again
            continue
        parts = []
        for mask in (left, ~left):
            sub = idx[mask]; m_ = Xi[mask].mean(0)
            parts.append((sub, m_, float(sq[sub].sum() - len(sub) * m_ @ m_)))
        clusters[j], mus[j], sse[j] = parts[0]
        clusters.append(parts[1][0]); mus.append(parts[1][1]); sse.append(parts[1][2])
    C = np.array(mus)
    if len(C) < k:
        C = np.vstack([C, np.repeat(mu0[None], k - len(C), 0)])
    return C


def pca_part(X, k):
    return divisive_part(X, k, "pca")


def var_part(X, k):
    return divisive_part(X, k, "var")


In [ ]:
%%writefile /content/ockm_code/ockm_fast.py
"""OCKM-fast: a fast, deterministic version of orbit-chart (SAC-CKM / OCKM) seeding.

It keeps the OCKM idea (orbit charts E, H, P on the principal plane, equal-count
bands of the invariant I and sectors of the orbit coordinate s, seeds chosen by
density-filtered farthest-first, DFF) and changes only how each step is computed:

1. principal plane by randomised subspace iteration (3 steps) on a fixed
   subsample of at most 10,000 points, instead of the d x d covariance of all points;
2. one pass over the data gives the chart coordinates and a fixed Gaussian
   projection of dimension p = min(d, 128); cell centroids are formed in the
   projection with one sparse product, and full-dimensional centroids are
   computed only for the k chosen cells;
3. DFF on squared distances in the projection, one matrix-vector product per
   step;
4. the chart is chosen by the seeding cost on a fixed 5,000-point subsample,
   measured in the same projection (instead of three full cost passes, or
   three Lloyd runs as in OCKM's min3 rule), so one Lloyd run follows.

Variant 'split': the complement sign split of OCKM, but on r extra principal
coordinates only, with r = floor(log2(n / (8 k t))) clipped to [0, 8] and
t = 20 the target mean occupancy. This keeps cells populated (the full split
makes one cell per point when d is large). For k >= n / (8 t) it equals the
plain version.

Everything uses a fixed random seed, so the output is deterministic.
"""
from __future__ import annotations
import math
import numpy as np
from scipy import sparse
import sacckm

P_DIM = 128
N_SUB = 5000
T_OCC = 20


def rand_pcs(X, mu, r, seed=0, oversample=8, iters=4):
    """Top-r principal directions (r x d) of X - mu by randomised subspace iteration."""
    n, d = X.shape
    l = min(d, r + oversample)
    rng = np.random.default_rng(seed)
    G = rng.standard_normal((d, l))
    Y = X @ G - mu @ G
    for _ in range(iters):
        Q, _ = np.linalg.qr(Y)
        Z = X.T @ Q - np.outer(mu, Q.sum(0))          # (X - mu)^T Q
        Q2, _ = np.linalg.qr(Z)
        Y = X @ Q2 - mu @ Q2
    Q, _ = np.linalg.qr(Y)
    Bt = X.T @ Q - np.outer(mu, Q.sum(0))              # d x l = (Q^T (X - mu))^T
    _, _, Vt = np.linalg.svd(Bt.T, full_matrices=False)
    W = Vt[:r].copy()
    for j in range(r):                                 # sign convention of sacckm.pc_plane
        i = np.argmax(np.abs(W[j]))
        if W[j, i] < 0:
            W[j] *= -1
    return W


def _dff(Cp, counts, k, expected):
    """Density-filtered farthest-first on the rows of Cp (squared distances)."""
    m = len(Cp)
    chosen = [int(np.argmax(counts))]
    if m == 1:
        return chosen
    sq = np.einsum("ij,ij->i", Cp, Cp)
    sep = np.maximum(sq - 2.0 * Cp @ Cp[chosen[0]] + sq[chosen[0]], 0.0)
    pool = counts >= expected
    taken = np.zeros(m, bool); taken[chosen[0]] = True
    while len(chosen) < min(k, m):
        cand = pool & ~taken
        if not cand.any():
            cand = ~taken
        score = np.where(cand, sep, -np.inf)
        j = int(np.argmax(score)); chosen.append(j); taken[j] = True
        np.minimum(sep, np.maximum(sq - 2.0 * Cp @ Cp[j] + sq[j], 0.0), out=sep)
    return chosen


def _cost_proj(Sp, Cp):
    d2 = np.einsum("ij,ij->i", Sp, Sp)[:, None] + np.einsum("ij,ij->i", Cp, Cp)[None, :] - 2.0 * Sp @ Cp.T
    return float(np.maximum(d2.min(1), 0).sum())


def ockm_fast(X, k, variant="plain", Q=4, seed=0, return_info=False, n_pca=10000, pca_iters=3):
    n, d = X.shape
    mu = X.mean(0)
    B = k
    r_extra = 0
    if variant == "split":
        r_extra = int(np.clip(math.floor(math.log2(max(n / (8.0 * k * T_OCC), 1.0))), 0, 8))
    rng = np.random.default_rng(seed + 1)
    # principal directions from a fixed subsample (PCA only needs the plane, not every point)
    ps = rng.choice(n, n_pca, replace=False) if n > n_pca else np.arange(n)
    Xs = X[ps]
    W = rand_pcs(Xs, Xs.mean(0), 2 + r_extra, seed=seed, iters=pca_iters)
    p = min(d, P_DIM)
    G = rng.standard_normal((d, p)) / math.sqrt(p) if p < d else np.eye(d)
    # one pass over X: chart coordinates and the random projection together
    A = np.hstack([W.T, G])
    XA = X @ A - mu @ A
    UV, Xp = XA[:, :2 + r_extra], XA[:, 2 + r_extra:]
    u, v = UV[:, 0], UV[:, 1]
    sub = rng.choice(n, min(n, N_SUB), replace=False)
    Sp = Xp[sub]
    extra_key = None
    if r_extra:
        bits = (UV[:, 2:] > 0).astype(np.int64)
        extra_key = (bits * (1 << np.arange(r_extra))).sum(1)
    best = None
    for regime in sacckm.REGIMES:
        I, s, lab = sacckm.chart(u, v, regime)
        key = (sacckm.quantile_bins(I, B) * (Q * 2) + sacckm.quantile_bins(s, Q) * 2 + lab).astype(np.int64)
        if extra_key is not None:
            key = key * (1 << r_extra) + extra_key
        uniq, inv, counts = np.unique(key, return_inverse=True, return_counts=True)
        inv = inv.ravel(); m = len(uniq)
        M = sparse.csr_matrix((np.ones(n), (inv, np.arange(n))), shape=(m, n))
        Cp = np.asarray(M @ Xp) / counts[:, None]        # centred projected cell centroids
        expected = counts.mean() if r_extra else n / (B * Q)
        ch = _dff(Cp, counts, k, expected)
        cost = _cost_proj(Sp, Cp[ch])
        if best is None or cost < best[0]:
            best = (cost, regime, M[ch], counts[ch], m)
    cost, regime, Mch, cch, m = best
    centres = np.asarray(Mch @ X) / cch[:, None]          # full-dimensional centroids of the chosen cells only
    if len(centres) < k:
        centres = np.vstack([centres, np.repeat(mu[None], k - len(centres), 0)])
    if return_info:
        return centres, dict(regime=regime, n_cells=m, r_extra=r_extra)
    return centres


In [ ]:
%%writefile /content/ockm_code/sacckm.py
"""
SAC-CKM prototype: Symmetry-Adaptive Concentric-region K-Means seeding
with corrected one-parameter-subgroup charts.

Everything (densities, centroids, Lloyd, metrics) is computed in the
standardised d-dimensional feature space. The one-parameter subgroup acts on
the plane spanned by the first two principal directions of the standardised
data and trivially on the orthogonal complement. That plane is used ONLY to
define the invariant coordinate I(x) and the orbit coordinate s(x).

Charts (u, v) = coordinates in the PC plane, centred at the global centroid:

  regime      generator A        flow T(t)            I(x)            s(x)
  elliptic    [[0,-1],[1,0]]     rotation by t        sqrt(u²+v²)     atan2(v,u)
  hyperbolic  [[1,0],[0,-1]]     (e^t u, e^-t v)      u*v             ½ ln(|u|/|v|)  (+ sign(u) label)
  parabolic   [[0,1],[0,0]]      (u + t v, v)         v               u / v

Along the flow, I is constant and s shifts by exactly t. Bands are
equal-count (quantile) intervals of I, sectors are equal-count intervals of s,
so the whole B x Q partition is shifted rigidly by the flow: the seeding is
equivariant under the chosen subgroup.

CCKM (Algorithm 1 of the manuscript) is implemented verbatim for comparison.

[Rebuilt from the Drive copy of ockm/sacckm.py (OCKM repo, 16 Sep 2026).
 Functional code unchanged. The only addition is the optional argument
 `hash_signs` of sac_init_regime / sac_min3 / sac_init_subsample: when True the
 complement sign pattern is hashed with np.unique on packed bits instead of the
 integer shift `key * 2**d`, which overflows int64 for d > 62 (the original
 code assumes d <= 62). For d <= 62 both give the same partition.]
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional

EPS = 1e-12


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def load_dataset(path: str):
    df = pd.read_csv(path)
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    label_col = [c for c in df.columns if c.lower() == "class"][0]
    y = pd.factorize(df[label_col].astype(str))[0]
    X = df.drop(columns=[label_col]).to_numpy(dtype=float)
    return X, y


def standardise(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0)
    sd[sd == 0] = 1.0
    return (X - mu) / sd


# ----------------------------------------------------------------------------
# CCKM (Algorithm 1 of the manuscript, d-dimensional sign regions)
# ----------------------------------------------------------------------------
def _sign_key(v: np.ndarray) -> tuple:
    # non-positive coordinate (0) precedes positive (1); first coordinate primary
    return tuple((v > 0).astype(int).tolist())


def cckm_init(D: np.ndarray, k: int) -> dict:
    n, d = D.shape
    C = D.mean(axis=0)
    dist = np.linalg.norm(D - C, axis=1)
    r_min, r_max = dist.min(), dist.max()
    h = (r_max - r_min) / k
    centres = np.empty((k, d))
    info = {"empty_annuli": 0}
    if h == 0:
        # degenerate radial range: rank sign regions of D by cardinality
        keys = [_sign_key(x - C) for x in D]
        uniq, counts = np.unique(np.array(keys), axis=0, return_counts=True)
        order = np.lexsort((*[uniq[:, j] for j in range(d - 1, -1, -1)], -counts))
        regs = uniq[order]
        for i in range(k):
            if i < len(regs):
                m = np.all(np.array(keys) == regs[i], axis=1)
                centres[i] = D[m].mean(axis=0)
            else:
                centres[i] = C
        return {"centres": centres, **info}
    r = r_min + h * np.arange(k + 1)
    for i in range(1, k + 1):
        m = (dist > r[i - 1]) & (dist <= r[i])
        A = D[m]
        if len(A) == 0:
            centres[i - 1] = C
            info["empty_annuli"] += 1
            continue
        keys = (A - C > 0).astype(int)
        uniq, counts = np.unique(keys, axis=0, return_counts=True)
        best = counts.max()
        cand = uniq[counts == best]
        # ties: lexicographic order of sign pattern, non-positive first, first coord primary
        cand = cand[np.lexsort([cand[:, j] for j in range(d - 1, -1, -1)])]
        tq = cand[0]
        mm = np.all(keys == tq, axis=1)
        centres[i - 1] = A[mm].mean(axis=0)
    return {"centres": centres, **info}


# ----------------------------------------------------------------------------
# SAC-CKM
# ----------------------------------------------------------------------------
REGIMES = ("E", "H", "P")


def pc_plane(D: np.ndarray):
    """Return (u, v) coordinates in the first two principal directions of the
    centred standardised data, and the 2 x d basis."""
    C = D.mean(axis=0)
    Z = D - C
    if D.shape[1] == 1:
        Z = np.hstack([Z, np.zeros_like(Z)])
    # principal directions from the d x d covariance (O(n d^2), no n x d factor);
    # deterministic sign convention (largest |loading| positive)
    evals, evecs = np.linalg.eigh(Z.T @ Z)
    W = evecs[:, ::-1][:, :2].T.copy()
    for j in range(2):
        i = np.argmax(np.abs(W[j]))
        if W[j, i] < 0:
            W[j] *= -1
    uv = Z @ W.T
    return uv[:, 0], uv[:, 1], W, C


def chart(u: np.ndarray, v: np.ndarray, regime: str):
    """Invariant coordinate I, orbit coordinate s, and discrete sign label
    (for the hyperbolic regime, where a level set of I=uv has two branches)."""
    if regime == "E":
        I = np.sqrt(u * u + v * v)
        s = np.arctan2(v, u)
        lab = np.zeros_like(u, dtype=int)
    elif regime == "H":
        I = u * v
        s = 0.5 * np.log((np.abs(u) + EPS) / (np.abs(v) + EPS))
        lab = (u > 0).astype(int)
    elif regime == "P":
        I = v
        denom = np.where(np.abs(v) < EPS, np.where(v < 0, -EPS, EPS), v)
        s = u / denom
        lab = np.zeros_like(u, dtype=int)
    else:
        raise ValueError(regime)
    return I, s, lab


def quantile_bins(x: np.ndarray, nb: int) -> np.ndarray:
    """Equal-count bins by rank (ties broken by index, deterministic)."""
    order = np.argsort(x, kind="stable")
    b = np.empty(len(x), dtype=int)
    b[order] = (np.arange(len(x)) * nb) // len(x)
    return b


@dataclass
class SACResult:
    centres: np.ndarray
    regime: str
    init_sse: float
    n_cells: int
    cell_counts: dict = field(default_factory=dict)
    seeds_cells: list = field(default_factory=list)
    sse_by_regime: dict = field(default_factory=dict)
    chart_data: Optional[dict] = None


def _sse(D: np.ndarray, centres: np.ndarray) -> float:
    # memory-safe: ||x||^2 + ||c||^2 - 2 x.c, in blocks of 100k points
    tot = 0.0; cn = (centres ** 2).sum(1)
    for i in range(0, len(D), 100_000):
        X = D[i:i + 100_000]
        d2 = (X ** 2).sum(1)[:, None] + cn[None, :] - 2.0 * X @ centres.T
        tot += float(np.maximum(d2.min(axis=1), 0).sum())
    return tot


def _select_seeds(cands: np.ndarray, counts: np.ndarray, k: int, expected: float,
                  mode: str = "dff", alpha: float = 0.5):
    """Deterministic seed selection from candidate cell centroids.

    mode = 'dff'  : density-filtered farthest-first (parameter-free).
                    Start from the densest cell; then repeatedly take, among cells
                    with count >= expected (the mean cell occupancy), the one
                    farthest from the already selected seeds. If the dense pool is
                    exhausted, continue farthest-first over all remaining cells.
    mode = 'score': alpha * density~ + (1-alpha) * separation~ (the weighted rule
                    from the original SAC-CKM proposal), kept for ablation.
    """
    m = len(cands)
    chosen = [int(np.argmax(counts))]
    if m == 1:
        return chosen
    dens = counts / counts.max()
    diam = (np.max(np.linalg.norm(cands[:, None] - cands[None, :], axis=-1)) + EPS) if mode == "score" else 1.0
    sep = np.linalg.norm(cands - cands[chosen[0]], axis=1)  # running min distance to chosen seeds
    while len(chosen) < min(k, m):
        sep = sep.copy()
        sep[chosen] = -np.inf
        if mode == "dff":
            pool = (counts >= expected)
            pool[chosen] = False
            if not pool.any():
                pool = np.ones(m, bool); pool[chosen] = False
            score = np.where(pool, sep, -np.inf)
        else:
            score = alpha * dens + (1 - alpha) * sep / diam
            score[chosen] = -np.inf
        j = int(np.argmax(score)); chosen.append(j)
        sep = np.minimum(sep, np.linalg.norm(cands - cands[j], axis=1))
    return chosen


def _complement_key(key: np.ndarray, sign: np.ndarray, hash_signs: bool) -> np.ndarray:
    n, d = sign.shape
    if not hash_signs:
        sk = np.zeros(n, dtype=np.int64)
        for j in range(d):
            sk = sk * 2 + sign[:, j]
        return key * (2 ** d) + sk
    # overflow-free: label each distinct packed sign row, then combine with key
    packed = np.packbits(sign.astype(np.uint8), axis=1)
    _, sid = np.unique(packed, axis=0, return_inverse=True)
    sid = sid.ravel().astype(np.int64)
    return key * (int(sid.max()) + 1) + sid


def sac_init_regime(D: np.ndarray, k: int, regime: str, B: Optional[int] = None,
                    Q: int = 4, mode: str = "dff", alpha: float = 0.5,
                    keep_chart: bool = False, complement: bool = False,
                    density_ref: str = "auto", plane=None, compute_sse: bool = True,
                    hash_signs: bool = False) -> SACResult:
    """complement=True additionally splits every (band, sector) cell by the sign
    pattern of the coordinates orthogonal to the PC plane (in the original
    standardised axes, as CCKM does). The subgroup acts trivially there, so the
    partition stays equivariant while regaining d-dimensional resolution."""
    n, d = D.shape
    B = k if B is None else B
    u, v, W, C = pc_plane(D) if plane is None else plane
    I, s, lab = chart(u, v, regime)
    bI = quantile_bins(I, B)
    bS = quantile_bins(s, Q)
    key = (bI * (Q * 2) + bS * 2 + lab).astype(np.int64)
    if complement and d > 2:
        Z = D - C
        Zc = Z - (Z @ W.T) @ W  # component orthogonal to the PC plane
        sign = (Zc > 1e-12).astype(np.int64)
        key = _complement_key(key, sign, hash_signs or d > 62)
    uniq, inv, counts = np.unique(key, return_inverse=True, return_counts=True)
    inv = inv.ravel()
    cands = np.stack([np.bincount(inv, weights=D[:, j], minlength=len(uniq)) for j in range(d)], axis=1) / counts[:, None]
    # density reference: expected occupancy under independence of the I and s
    # bins for the plain quantile grid; mean occupancy of populated cells when the
    # complement split makes the number of possible cells data-dependent.
    # density_ref: 'indep' = n/(B*Q); 'popmean' = mean occupancy of populated cells;
    # 'auto' = popmean when the complement split is active (d > 2), else indep.
    if density_ref == "auto":
        density_ref = "popmean" if (complement and d > 2) else "indep"
    expected = counts.mean() if density_ref == "popmean" else n / (B * Q)
    chosen = _select_seeds(cands, counts, k, expected, mode, alpha)
    centres = cands[chosen]
    if len(centres) < k:  # fewer populated cells than k: pad with global centroid
        centres = np.vstack([centres, np.repeat(C[None], k - len(centres), 0)])
    res = SACResult(centres=centres, regime=regime, init_sse=(_sse(D, centres) if compute_sse else np.nan),
                    n_cells=len(uniq), seeds_cells=[int(uniq[j]) for j in chosen],
                    cell_counts={int(uniq[j]): int(counts[j]) for j in range(len(uniq))})
    if keep_chart:
        res.chart_data = dict(u=u, v=v, I=I, s=s, lab=lab, bI=bI, bS=bS, W=W, C=C,
                              cell_key=key, cand=cands, chosen=chosen)
    return res


def cckm_cells_init(D: np.ndarray, k: int, bands: str = "equal", mode: str = "dff") -> SACResult:
    """Ablation: CCKM's own d-dimensional cells (annulus x sign region about the
    global centroid) but with the SAC selection rule instead of one-seed-per-annulus.
    bands = 'equal'    : k equal-width annuli exactly as CCKM (empty annuli possible)
    bands = 'quantile' : k equal-count radial bands
    No PCA plane, no subgroup chart: isolates the effect of the selection step."""
    n, d = D.shape
    C = D.mean(axis=0)
    Z = D - C
    dist = np.linalg.norm(Z, axis=1)
    if bands == "equal":
        r_min, r_max = dist.min(), dist.max()
        h = (r_max - r_min) / k
        if h == 0:
            b = np.zeros(n, int)
        else:
            b = np.clip(np.ceil((dist - r_min) / h).astype(int) - 1, 0, k - 1)
            b[dist <= r_min] = 0  # innermost point joins band 1
    else:
        b = quantile_bins(dist, k)
    sign = (Z > 0).astype(np.int64)
    # hash the sign pattern per point
    sk = np.zeros(n, dtype=np.int64)
    for j in range(d):
        sk = sk * 2 + sign[:, j]  # d <= 62 assumed
    key = b.astype(np.int64) * (2 ** d) + sk
    uniq, inv, counts = np.unique(key, return_inverse=True, return_counts=True)
    cands = np.stack([np.bincount(inv, weights=D[:, j], minlength=len(uniq)) for j in range(d)], axis=1) / counts[:, None]
    expected = counts.mean()  # mean occupancy of the populated cells
    chosen = _select_seeds(cands, counts, k, expected, mode)
    centres = cands[chosen]
    if len(centres) < k:
        centres = np.vstack([centres, np.repeat(C[None], k - len(centres), 0)])
    return SACResult(centres=centres, regime="cckm-" + bands, init_sse=_sse(D, centres),
                     n_cells=len(uniq), seeds_cells=[int(uniq[j]) for j in chosen])


def sac_init(D: np.ndarray, k: int, regime: str = "auto", **kw) -> SACResult:
    """regime in {'auto','E','H','P'}. 'auto' evaluates the three charts and keeps
    the seeding with the lowest initial SSE (no Lloyd, no threshold)."""
    if regime != "auto":
        return sac_init_regime(D, k, regime, **kw)
    kw.setdefault("plane", pc_plane(D))
    results = {q: sac_init_regime(D, k, q, **kw) for q in REGIMES}
    best = min(REGIMES, key=lambda q: results[q].init_sse)  # ties -> E, H, P order
    out = results[best]
    out.sse_by_regime = {q: results[q].init_sse for q in REGIMES}
    return out


def sac_init_subsample(D: np.ndarray, k: int, n_sub: int = 5000, seed: int = 0, complement: bool = True, **kw):
    """Scalable regime selection: run Lloyd from the three charts on a fixed random
    subsample of n_sub points, keep the chart with the lowest subsample WCSS, then
    return the full-data seeding of that chart (one full Lloyd run follows)."""
    from sklearn.cluster import KMeans
    n = len(D)
    if n <= n_sub:
        return None  # caller should use the plain min3 rule
    idx = np.random.default_rng(seed).choice(n, n_sub, replace=False)
    Ds = D[idx]
    best = None
    for q in REGIMES:
        r = sac_init_regime(Ds, k, q, complement=complement, compute_sse=False, **kw)
        km = KMeans(n_clusters=k, init=r.centres, n_init=1, max_iter=300, tol=1e-4, algorithm="lloyd").fit(Ds)
        if best is None or km.inertia_ < best[1]:
            best = (q, km.inertia_)
    return sac_init_regime(D, k, best[0], complement=complement, compute_sse=False, **kw)


def sac_min3(D: np.ndarray, k: int, lloyd_fn, complement: bool = True, **kw):
    """Run Lloyd from the three charts (PC plane computed once) and return
    (result, fitted_model, init_time_seconds) of the lowest final WCSS."""
    import time as _t
    t0 = _t.perf_counter(); plane = pc_plane(D); ti = _t.perf_counter() - t0
    best = None
    for q in REGIMES:
        ta = _t.perf_counter(); r = sac_init_regime(D, k, q, complement=complement, plane=plane, compute_sse=False, **kw); ti += _t.perf_counter() - ta
        km = lloyd_fn(D, k, r.centres)
        if best is None or km.inertia_ < best[1].inertia_: best = (r, km)
    return best[0], best[1], ti


In [ ]:
%%writefile /content/heldout_bench.py
"""Held-out large-scale benchmark for OCKM-T (none of these data sets was used for tuning).

Protocol (same as the paper): every seeding is followed by one Lloyd run with scikit-learn
(n_init=1, algorithm='lloyd', tol=1e-4); randomised methods use seeds 0..R-1; the gap of a run is
its final cost above the lowest final cost reached by any run on that setting.
Rows are appended to a CSV as soon as they finish, so the run can be resumed in a new session.
"""
from __future__ import annotations
import os, sys, time, json, warnings
import numpy as np, pandas as pd
from sklearn.cluster import KMeans, kmeans_plusplus
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------- data sets
def _standardise(X):
    X = np.asarray(X, dtype=np.float64)
    X = X[:, X.std(0) > 0]
    return np.ascontiguousarray((X - X.mean(0)) / X.std(0))


def _pca(X, dim, seed=0):
    from sklearn.decomposition import PCA
    return np.ascontiguousarray(PCA(dim, random_state=seed).fit_transform(X).astype(np.float64))


def _image_features(name, device):
    """ResNet-50 (ImageNet weights) penultimate features of the CIFAR training set, reduced to 128 dims by PCA."""
    import torch, torchvision
    from torchvision import transforms
    ds_cls = {"cifar10": torchvision.datasets.CIFAR10, "cifar100": torchvision.datasets.CIFAR100,
              "svhn": torchvision.datasets.SVHN}[name]
    tf = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    ds = ds_cls("/content/tv_data", split="train", download=True, transform=tf) if name == "svhn" else \
        ds_cls("/content/tv_data", train=True, download=True, transform=tf)
    y = np.array(ds.labels if name == "svhn" else ds.targets)
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = torch.nn.Identity(); model.eval().to(device)
    dl = torch.utils.data.DataLoader(ds, batch_size=256, num_workers=2)
    feats = []
    with torch.no_grad():
        for xb, _ in dl:
            feats.append(model(xb.to(device)).float().cpu().numpy())
    return _pca(np.concatenate(feats), 128), y


def _text_features(name, device):
    """all-MiniLM-L6-v2 sentence embeddings (384 dims, unit length)."""
    from sentence_transformers import SentenceTransformer
    if name == "20ng":
        from sklearn.datasets import fetch_20newsgroups
        d = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"))
        texts, y = d.data, np.array(d.target)
    else:  # ag_news
        from datasets import load_dataset
        d = None
        for repo in ("fancyzhx/ag_news", "ag_news"):          # the Hub moved 'ag_news' to 'fancyzhx/ag_news'
            try:
                d = load_dataset(repo, split="train"); break
            except Exception as e:
                print("could not load", repo, "->", type(e).__name__)
        if d is None:
            raise RuntimeError("AG News could not be downloaded from the Hugging Face Hub")
        texts, y = d["text"], np.array(d["label"])
    m = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    E = m.encode(texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
    return np.ascontiguousarray(E.astype(np.float64)), y


def load_dataset(name, cache_dir, device="cpu"):
    """Returns (X, y). Features are computed once and cached as .npz in cache_dir."""
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{name}.npz")
    if os.path.exists(path):
        z = np.load(path); return z["X"], z["y"]
    if name == "covtype":
        from sklearn.datasets import fetch_covtype
        d = fetch_covtype(); X, y = _standardise(d.data), d.target
    elif name in ("cifar10", "cifar100", "svhn"):
        X, y = _image_features(name, device)
    elif name in ("20ng", "agnews"):
        X, y = _text_features(name, device)
    elif name == "letter":   # only for a quick test; letter was already in the paper's large-k study
        from sklearn.datasets import fetch_openml
        d = fetch_openml("letter", version=1, as_frame=False); X, y = _standardise(d.data), d.target
    else:
        raise ValueError(name)
    np.savez_compressed(path, X=X, y=np.asarray(y))
    return X, np.asarray(y)


# --------------------------------------------------------------------------------- methods
def get_methods(code_dir):
    sys.path.insert(0, code_dir)
    import ockmt
    from ockmt import ockm_t
    from qkmeans import qkmeans
    from baselines2 import afkmc2, kmeans_par
    from det_baselines import pca_part, var_part
    import ockm_tc

    def _t(X, k, s, **kw):
        c, info = ockm_t(X, k, return_info=True, seed=s, **kw)
        return c, {kk: info[kk] for kk in ("phi", "phi_full", "n_starts", "m") if kk in info}

    methods = {
        "OCKM-T":    (lambda X, k, s: _t(X, k, s), True),
        "Circle tree": (lambda X, k, s: (ockm_tc.circle_tree_seeding(X, k, seed=s), {}), True),
        "T-pca":     (lambda X, k, s: _t(X, k, s, pca_only=True), True),
        "KM++":      (lambda X, k, s: (kmeans_plusplus(X, k, random_state=s)[0], {}), True),
        "KM++v":     (lambda X, k, s: (kmeans_plusplus(X, k, random_state=s, n_local_trials=1)[0], {}), True),
        "Qk":        (lambda X, k, s: (qkmeans(X, k, m=10.0, seed=s), {}), True),
        "AFK-MC2":   (lambda X, k, s: (afkmc2(X, k, seed=s), {}), True),
        "k-means||": (lambda X, k, s: (kmeans_par(X, k, seed=s), {}), True),
        "PCA-Part":  (lambda X, k, s: (pca_part(X, k), {}), False),
        "Var-Part":  (lambda X, k, s: (var_part(X, k), {}), False),
    }
    return methods, ockm_tc.warmup


def cost(X, C, chunk=8192):
    cc = (C * C).sum(1)
    tot = 0.0
    for i in range(0, len(X), chunk):
        A = X[i:i + chunk]
        D = (A * A).sum(1)[:, None] + cc[None, :] - 2.0 * A @ C.T
        tot += float(np.maximum(D.min(1), 0).sum())
    return tot


CORE = ("OCKM-T", "Circle tree", "KM++")      # methods that answer the main question: always all seeds


# ---------------------------------------------------------------------------- GPU Lloyd
def lloyd_torch(X, C, max_iter=300, tol=1e-4, device="cuda", chunk=65536):
    """Lloyd's algorithm on the GPU, same rules as scikit-learn's 'lloyd': stop when the
    assignment is unchanged or the total squared centre shift is at most tol * mean feature
    variance; an empty cluster is moved to the point farthest from its centre. Iterations run
    in float32; the final cost and labels are recomputed in float64 on the CPU."""
    import torch
    Xt = torch.as_tensor(X, dtype=torch.float32, device=device)
    Ct = torch.as_tensor(C, dtype=torch.float32, device=device).clone()
    n, k = Xt.shape[0], Ct.shape[0]
    tol_abs = tol * float(np.mean(np.var(X, axis=0)))
    xx = (Xt * Xt).sum(1)
    prev = None; it = 0
    for it in range(1, max_iter + 1):
        cc = (Ct * Ct).sum(1)
        lab = torch.empty(n, dtype=torch.long, device=device); dmin = torch.empty(n, device=device)
        for i in range(0, n, chunk):
            D = xx[i:i + chunk, None] + cc[None, :] - 2.0 * Xt[i:i + chunk] @ Ct.T
            dmin[i:i + chunk], lab[i:i + chunk] = D.min(1)
        if prev is not None and torch.equal(lab, prev):
            it -= 1; break
        cnt = torch.bincount(lab, minlength=k).float()
        S = torch.zeros_like(Ct).index_add_(0, lab, Xt)
        newC = Ct.clone(); nz = cnt > 0
        newC[nz] = S[nz] / cnt[nz, None]
        if (~nz).any():                                    # relocate empty clusters
            far = torch.argsort(dmin, descending=True)[: int((~nz).sum())]
            newC[~nz] = Xt[far]
        shift = float(((newC - Ct) ** 2).sum())
        Ct = newC; prev = lab
        if shift <= tol_abs:
            break
    Cf = Ct.double().cpu().numpy()
    labels = np.empty(n, dtype=np.int64); inertia = 0.0; cf2 = (Cf * Cf).sum(1)
    for i in range(0, n, 8192):
        A = X[i:i + 8192]
        D = (A * A).sum(1)[:, None] + cf2[None, :] - 2.0 * A @ Cf.T
        labels[i:i + 8192] = D.argmin(1); inertia += float(np.maximum(D.min(1), 0).sum())
    return Cf, labels, inertia, it


# ------------------------------------------------------------------------------ benchmark
def run_setting(name, k, runs, X, y, methods, out_csv, max_iter=300, only=None, other_runs=3, gpu_lloyd=False):
    """runs: seeds for the core methods; other_runs: seeds for the other randomised baselines."""
    done = set()
    if os.path.exists(out_csv):
        d = pd.read_csv(out_csv, usecols=["dataset", "k", "method", "seed"])
        done = set(zip(d.dataset, d.k, d.method, d.seed))
    for meth, (fn, randomised) in methods.items():
        if only and meth not in only:
            continue
        nr = runs if meth in CORE else min(runs, other_runs)
        for s in (range(nr) if randomised else [0]):
            if (name, k, meth, s) in done:
                continue
            t0 = time.perf_counter(); C, ex = fn(X, k, s); ts = time.perf_counter() - t0
            C = np.ascontiguousarray(C, dtype=np.float64)
            t0 = time.perf_counter()
            if gpu_lloyd:
                _, labs, wcss, n_it = lloyd_torch(X, C, max_iter=max_iter, device=gpu_lloyd if isinstance(gpu_lloyd, str) else "cuda")
            else:
                km = KMeans(k, init=C, n_init=1, max_iter=max_iter, tol=1e-4, algorithm="lloyd", random_state=0).fit(X)
                labs, wcss, n_it = km.labels_, float(km.inertia_), int(km.n_iter_)
            tl = time.perf_counter() - t0
            row = dict(dataset=name, n=len(X), d=X.shape[1], k=k, method=meth, seed=s, seed_time=ts,
                       seed_cost=cost(X, C), WCSS=float(wcss), IT=int(n_it), lloyd_time=tl, total_time=ts + tl,
                       hit_max_iter=int(n_it >= max_iter), lloyd="gpu" if gpu_lloyd else "sklearn",
                       ARI=adjusted_rand_score(y, labs), NMI=normalized_mutual_info_score(y, labs),
                       extra=json.dumps({kk: float(v) for kk, v in ex.items()}))
            pd.DataFrame([row]).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
        print(f"{name:10s} k={k:5d} {meth:10s} done", flush=True)


# -------------------------------------------------------------------------------- summary
def _holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj


def add_portfolios(d):
    """Two-run portfolios built from recorded runs, keeping the lower FINAL cost (two Lloyd runs):
    'Line+Circle (2 runs)' = OCKM-T seed s and circle tree seed s;
    'Line+Line (2 runs)'   = OCKM-T seeds s and s+1 (mod R), the fair comparison with the same work."""
    rows = []
    for (ds, k), D in d.groupby(["dataset", "k"]):
        L = D[D.method == "OCKM-T"].set_index("seed"); Cc = D[D.method == "Circle tree"].set_index("seed")
        seeds = sorted(set(L.index) & set(Cc.index)); R = len(L)
        for s in seeds:
            for name, a, b in (("Line+Circle (2 runs)", L.loc[s], Cc.loc[s]),
                               ("Line+Line (2 runs)", L.loc[s], L.loc[sorted(L.index)[(sorted(L.index).index(s) + 1) % R]])):
                win = a if a.WCSS <= b.WCSS else b
                r = win.copy(); r["method"] = name; r["seed"] = s
                r["total_time"] = a.total_time + b.total_time; r["seed_time"] = a.seed_time + b.seed_time
                rows.append(r)
            a, b = L.loc[s], Cc.loc[s]                      # OCKM-TC: pick by seeding cost, one Lloyd run
            win = a if a.seed_cost <= b.seed_cost else b
            r = win.copy(); r["method"] = "OCKM-TC"; r["seed"] = s
            r["seed_time"] = a.seed_time + b.seed_time; r["total_time"] = r["seed_time"] + win.lloyd_time
            rows.append(r)
    return pd.concat([d, pd.DataFrame(rows)], ignore_index=True) if rows else d


def read_results(out_csv):
    """out_csv may be one file or a glob pattern such as '.../results_part*.csv'."""
    import glob
    files = sorted(glob.glob(out_csv))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True).drop_duplicates(["dataset", "k", "method", "seed"])


def summarise(out_csv, ref="OCKM-T", tie=0.01, portfolios=True):
    from scipy import stats
    d = read_results(out_csv)
    if portfolios:
        d = add_portfolios(d)
    d["setting"] = d.dataset + " k=" + d.k.astype(str)
    best = d.groupby("setting").WCSS.min()
    d["gap"] = 100 * (d.WCSS / d.setting.map(best) - 1)
    g = d.groupby(["setting", "method"]).gap.mean().unstack()
    tm = d.groupby(["setting", "method"]).total_time.median().unstack()
    ranks = g.rank(axis=1).mean().sort_values()
    rows, pv = [], []
    for m in g.columns:
        if m == ref:
            continue
        x = (g[ref] - g[m]).dropna()
        w, t, l = int((x < -tie).sum()), int((x.abs() <= tie).sum()), int((x > tie).sum())
        p = stats.wilcoxon(x).pvalue if (x.abs() > 0).sum() >= 1 and len(x) >= 5 else np.nan
        rows.append(dict(method=m, mean_gap=g[m].mean(), mean_rank=ranks[m], time_vs_ref=(tm[m] / tm[ref]).median(),
                         **{f"{ref} better/tied/worse": f"{w}/{t}/{l}"}, p=p)); pv.append(p)
    tab = pd.DataFrame(rows)
    ok = ~tab.p.isna()
    tab.loc[ok, "p_holm"] = _holm(tab.p[ok].values)
    head = pd.DataFrame([dict(method=ref, mean_gap=g[ref].mean(), mean_rank=ranks[ref], time_vs_ref=1.0)])
    tab = pd.concat([head, tab], ignore_index=True).sort_values("mean_rank")
    try:
        from scipy.stats import friedmanchisquare
        cols = g.dropna().T.values
        fp = friedmanchisquare(*cols).pvalue if len(g.dropna()) >= 3 else np.nan
    except Exception:
        fp = np.nan
    return tab, g, fp


def equal_time(out_csv, ref="OCKM-T"):
    """For each run of ref, every randomised method is given the same total time: it is restarted with its
    recorded runs (in seed order) while the summed time stays within the budget (at least one run), and the
    best final cost is kept. Limited by the number of recorded runs."""
    d = add_portfolios(read_results(out_csv)); d = d[~d.method.isin(["Line+Circle (2 runs)", "Line+Line (2 runs)"])]
    d["setting"] = d.dataset + " k=" + d.k.astype(str)
    best = d.groupby("setting").WCSS.min(); res = []
    for st, D in d.groupby("setting"):
        R = D[D.method == ref]
        for m in D.method.unique():
            if m == ref or D[D.method == m].seed.nunique() <= 1:
                continue
            M = D[D.method == m].sort_values("seed")
            for _, r in R.iterrows():
                budget, acc, bestc = r.total_time, 0.0, np.inf
                order = np.roll(M.index.values, -int(r.seed) % len(M))
                for i in order:
                    if acc > 0 and acc + M.loc[i, "total_time"] > budget:
                        break
                    acc += M.loc[i, "total_time"]; bestc = min(bestc, M.loc[i, "WCSS"])
                res.append(dict(setting=st, method=m, ref_gap=100 * (r.WCSS / best[st] - 1), other_gap=100 * (bestc / best[st] - 1)))
    e = pd.DataFrame(res)
    if e.empty:
        return e
    e = e.groupby(["setting", "method"])[["ref_gap", "other_gap"]].mean().reset_index()
    e["diff"] = e.ref_gap - e.other_gap
    return e.groupby("method").apply(lambda x: pd.Series(dict(
        wtl=f"{int((x['diff'] < -0.01).sum())}/{int((x['diff'].abs() <= 0.01).sum())}/{int((x['diff'] > 0.01).sum())}",
        mean_ref_gap=x.ref_gap.mean(), mean_other_gap=x.other_gap.mean())))


In [ ]:
# 3. Settings: (data set, k, seeds for OCKM-T / circle tree / greedy k-means++), grouped into 3 parts.
#    Each part uses its own data sets, so two sessions never compute the same features.
PARTS = {
    1: [('cifar10', 10, 10), ('cifar10', 100, 10), ('cifar10', 1000, 5)],                  # 50,000 x 128 image features
    2: [('covtype', 7, 10), ('covtype', 100, 5), ('20ng', 20, 10), ('20ng', 200, 10)],        # tabular + text
    3: [('agnews', 4, 10), ('agnews', 100, 5), ('cifar100', 100, 10)],                        # text + image features
}
PART = 1              # 1, 2 or 3. Use 0 to run all three parts one after another in this session.
OTHER_RUNS = 3        # seeds for the other randomised baselines (KM++v, Qk, AFK-MC2, k-means||, T-pca)
GPU_LLOYD  = True     # Lloyd on the GPU (same rules as scikit-learn); False = scikit-learn on the CPU
QUICK_TEST = False    # True: one small setting with 2 seeds, to check that everything works

MY_SETTINGS = [s for p in sorted(PARTS) for s in PARTS[p]] if PART == 0 else PARTS[PART]
if QUICK_TEST:
    MY_SETTINGS = [('20ng', 20, 2)]
OUT_CSV = os.path.join(WORK_DIR, f'results_part{PART}.csv' if not QUICK_TEST else 'results_quicktest.csv')
print('this session runs:', MY_SETTINGS); print('results go to', OUT_CSV)

In [ ]:
# 4. Compute and cache the features once (uses the GPU)
import sys, torch, importlib
sys.path.insert(0, '/content')
import heldout_bench as hb; importlib.reload(hb)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if GPU_LLOYD and device != 'cuda':
    print('No GPU found: Lloyd will use scikit-learn on the CPU'); GPU_LLOYD = False
for name in sorted({s[0] for s in MY_SETTINGS}):
    X, y = hb.load_dataset(name, CACHE_DIR, device)
    print(f'{name:10s} n={X.shape[0]:7d} d={X.shape[1]:4d} classes={len(set(y))}')

In [ ]:
# 5. Run the benchmark (resumable). Numba compilation happens once and is not timed.
#    On Kaggle, download results_part*.csv from the Output tab when the run ends.
methods, warmup = hb.get_methods(CODE_DIR)
warmup()
if GPU_LLOYD:                                          # warm up the GPU so the first timing is fair
    import numpy as np; hb.lloyd_torch(np.random.rand(2000, 16), np.random.rand(10, 16))
import time
for name, k, runs in MY_SETTINGS:
    X, y = hb.load_dataset(name, CACHE_DIR, device)
    t0 = time.time()
    hb.run_setting(name, k, runs, X, y, methods, OUT_CSV, other_runs=OTHER_RUNS, gpu_lloyd=GPU_LLOYD)
    print(f'--- {name} k={k} finished in {(time.time()-t0)/60:.1f} min')

## Results
Run the cells below in any session once all parts have finished; they read every `results_part*.csv` file in the working folder.

In [ ]:
# 6. Combine the parts
import pandas as pd, glob
pd.set_option('display.width', 200)
ALL = OUT_CSV if QUICK_TEST else os.path.join(WORK_DIR, 'results_part*.csv')   # on Colab, first copy the Kaggle result files into this folder
d = hb.read_results(ALL)
print(len(d), 'runs from', len(glob.glob(ALL)), 'files'); display(d.groupby(['dataset', 'k']).method.nunique())

In [ ]:
# 7. Mean gap, mean rank, time relative to OCKM-T, OCKM-T better/tied/worse, Wilcoxon with Holm correction.
#    'Line+Circle (2 runs)' and 'Line+Line (2 runs)' keep the lower final cost of two runs (same work);
#    'OCKM-TC' picks between OCKM-T and the circle tree by seeding cost (one Lloyd run).
tab, per_setting, friedman_p = hb.summarise(ALL)
print('Friedman p =', friedman_p)
display(tab.round(3))
print('Mean gap (%) per setting:')
display(per_setting.round(3))

In [ ]:
# 8. Equal time: each randomised method gets OCKM-T's time and keeps its best restart
eq = hb.equal_time(ALL)
display(eq.round(3))

In [ ]:
# 9. Checks a reviewer will ask about
print('Runs that stopped at the 300-iteration cap:')
display(d.groupby(['dataset', 'k', 'method']).hit_max_iter.sum().loc[lambda s: s > 0])
print('Median seeding time (s):')
display(d.groupby(['dataset', 'k', 'method']).seed_time.median().unstack().round(2))
print('Mean ARI at k = number of classes:')
cls = {'20ng': 20, 'agnews': 4, 'cifar10': 10, 'cifar100': 100, 'covtype': 7}
display(d[d.k == d.dataset.map(cls)].groupby(['dataset', 'method']).ARI.mean().unstack().round(3))

In [ ]:
# 10. Save LaTeX tables and the combined CSV for the paper
d.to_csv(os.path.join(WORK_DIR, 'results_heldout_all.csv'), index=False)
tab.round(3).to_latex(os.path.join(WORK_DIR, 'table_heldout.tex'), index=False)
per_setting.round(2).to_latex(os.path.join(WORK_DIR, 'table_heldout_per_setting.tex'))
if len(eq):
    eq.round(3).to_latex(os.path.join(WORK_DIR, 'table_heldout_equal_time.tex'))
print('saved in', WORK_DIR)